# GS 2027 — Fábrica de Corpus CitizenTech
### Tema: Código de Defesa do Consumidor (CDC)

**Integrantes**
- Igor Paixão Sarak (RM 563726)
- Lucca Phelipe Masini (RM 564121)
- Bernardo Braga Perobeli (RM 562468)

---

**O que este notebook faz.** Transforma documentos públicos sobre direitos do consumidor em
corpus prontos para o fine-tuning de um modelo de linguagem pequeno (SLM) de atendimento ao cidadão:
ingestão → extração → limpeza → normalização → anonimização → segmentação → geração dos 3 corpus
e do conjunto de teste → relatório de qualidade.

**Como executar.** Ambiente do Colab com GPU **T4** (menu *Ambiente de execução → Alterar tipo de
ambiente de execução → T4 GPU*) e *Executar tudo*. Não precisa de chave de API nem de anexos: os
documentos são baixados de uma pasta pública do Google Drive (com cópia de reserva no GitHub).
A parte que usa o modelo de linguagem (seções 9 e 10) roda o **Qwen2.5-7B-Instruct** em 4 bits: o download
do modelo (cerca de 15 GB) e a geração levam, no total, de 45 a 60 minutos na T4.

**Arquivos gerados** (pasta `/content/gs_nlp/saida`): `corpus_texto.jsonl`, `corpus_qa.jsonl`,
`corpus_chat.jsonl`, `corpus_teste.jsonl`, `revisao.csv`, `amostra_fidelidade.csv`.

## Problema, persona e ODS

- **Persona:** consumidor com pouca escolaridade, que usa o celular e tem dúvida sobre cobrança
  indevida, troca de produto, cancelamento ou compra online.
- **Problema:** não sabe quais são os seus direitos nem a quem recorrer, e recebe atendimento
  genérico ou desatualizado de modelos que dependem da nuvem.
- **ODS 16** — Paz, justiça e instituições eficazes (metas 16.3, acesso à justiça, e 16.6,
  instituições eficazes e transparentes).

## Informações estáveis × voláteis

O treino ensina bem o **tom, o formato, o vocabulário e quando recusar ou encaminhar**; ensina mal
fatos que mudam. Por isso:

| Estáveis (podem ser aprendidas no treino) | Voláteis (devem ficar no RAG, com data de vigência) |
|---|---|
| Direitos básicos (art. 6º), conceitos de consumidor e fornecedor, vício × defeito, publicidade enganosa e abusiva, cláusulas abusivas, tom de acolhimento, quando encaminhar (Procon, Defensoria, advogado) | Prazos e regras do SAC (Decreto 13.135/2026, em vigor desde 02/11/2026 e que revogou o Decreto 11.034/2022), valor da multa do art. 57 (alterado pela MP 1.393/2026), orientações do guia da Black Friday de 2024 |

O corpus de **texto puro**, com fonte, seção e data, é a base pronta para o RAG; os corpus de
perguntas e respostas e de conversa ensinam o comportamento de atendente.

## 0. Instalação e configuração

In [1]:
import subprocess
import sys


def pip_install(packages):
    """Instala pacotes silenciosamente (apenas no Colab)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


# [COLAB-ONLY]
pip_install(["pypdf", "python-docx", "beautifulsoup4", "lxml", "ftfy", "gdown",
             "spacy", "sentence-transformers", "accelerate", "bitsandbytes"])
subprocess.run([sys.executable, "-m", "spacy", "download", "pt_core_news_lg", "-q"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'spacy', 'download', 'pt_core_news_lg', '-q'], returncode=0)

In [2]:
import hashlib
import html
import json
import os
import random
import re
import shutil
import time
import unicodedata
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)

# Pastas de trabalho. A variável de ambiente é opcional e serve só para testes locais.
WORK_DIR = Path(os.environ.get("GS_WORK_DIR", "/content/gs_nlp"))
INPUT_DIR = WORK_DIR / "entrada"
OUT_DIR = WORK_DIR / "saida"
CACHE_DIR = OUT_DIR / "geracao_cache"
for folder in (INPUT_DIR, OUT_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Opcional: guardar o cache da geração no seu Drive, para retomar se a sessão do Colab cair.
USE_DRIVE_CACHE = False
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/gs_nlp_cache")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Fonte dos documentos: pasta pública do Drive, com cópia de reserva no GitHub.
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1RglR7qdGky45VmmVZcATFcei15yb_ach?usp=sharing"
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/entregas-faculdade/GS-NLP-2S/main/docs_entrada/drive/"

# Modelos (todos abertos, sem chave de API).
TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B-Instruct"      # só para contar tokens, como na aula de fine-tuning
GENERATOR_NAME = "Qwen/Qwen2.5-7B-Instruct"        # geração de perguntas, respostas e diálogos (4 bits)
EMBEDDING_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
SPACY_MODEL = "pt_core_news_lg"

# Parâmetros do enunciado e metas de volume.
MIN_TOKENS, MAX_TOKENS = 256, 512
TARGET_QA, TARGET_CHAT = 130, 36          # mínimos do enunciado: 100 e 30
INFORMAL_RATE = 0.45                       # fração de perguntas "sujas" (mínimo do enunciado: 30%)
NEAR_DUP_THRESHOLD = 0.90                  # similaridade de cosseno para quase-duplicatas
LEAK_THRESHOLD = 0.88                      # similaridade com o teste que consideramos vazamento

print("Pasta de trabalho:", WORK_DIR)

Pasta de trabalho: /content/gs_nlp


## 1. Ingestão de documentos e metadados

Usamos **5 documentos oficiais** (4 formatos diferentes) e **1 documento-teste fictício**, que serve
só para demonstrar a anonimização e **não entra nos corpus finais**.

Para cada documento registramos: nome do arquivo, título, órgão emissor, link de origem e data de
publicação ou de vigência. Os arquivos são baixados da pasta pública do Drive; se o Drive falhar,
o notebook usa as cópias do GitHub.

In [3]:
DOCS = {
    "cdc_lei_8078_1990.html": {
        "title": "Lei nº 8.078/1990 — Código de Defesa do Consumidor (texto compilado)",
        "agency": "Presidência da República",
        "link": "https://www.planalto.gov.br/ccivil_03/Leis/L8078compilado.htm",
        "date": "1990-09", "accessed": "2026-11-11", "structure": "legal", "is_test": False,
        "volatile": "parcial (art. 57 alterado pela MP 1.393/2026)",
    },
    "lei_14181_2021.html": {
        "title": "Lei nº 14.181/2021 — Superendividamento (altera o CDC)",
        "agency": "Presidência da República",
        "link": "https://www.planalto.gov.br/ccivil_03/_ato2019-2022/2021/lei/L14181.htm",
        "date": "2021-07", "accessed": "2026-11-11", "structure": "legal", "is_test": False,
        "volatile": "não",
    },
    "decreto_13135_2026.docx": {
        "title": "Decreto nº 13.135/2026 — Serviço de Atendimento ao Consumidor (SAC)",
        "agency": "Presidência da República",
        "link": "https://www.planalto.gov.br/ccivil_03/_Ato2023-2026/2026/Decreto/D13135.htm",
        "date": "2026-10", "validity": "2026-11-02", "accessed": "2026-11-11", "structure": "legal",
        "is_test": False, "volatile": "sim (substituiu o Decreto 11.034/2022)",
    },
    "decreto_7962_2013.txt": {
        "title": "Decreto nº 7.962/2013 — Comércio eletrônico",
        "agency": "Presidência da República",
        "link": "https://www.planalto.gov.br/ccivil_03/_ato2011-2014/2013/decreto/d7962.htm",
        "date": "2013-03", "accessed": "2026-11-11", "structure": "legal", "is_test": False,
        "volatile": "não",
    },
    "guia_senacon_black_friday.pdf": {
        "title": "Guia do Consumidor para a Black Friday",
        "agency": "Ministério da Justiça e Segurança Pública / Secretaria Nacional do Consumidor (Senacon)",
        "link": ("https://www.gov.br/mj/pt-br/assuntos/noticias/ministerio-da-justica-lanca-guia-do-"
                 "consumidor-para-a-black-friday/copy2_of_CartilhaSenaconBlackFriday12.pdf"),
        "date": "2024-11", "accessed": "2026-11-11", "structure": "guia", "is_test": False,
        "volatile": "sim (edição de 2024)",
    },
    "documento_teste_ficticio.txt": {
        "title": "Ficha de atendimento fictícia (teste de anonimização)",
        "agency": "Fictício — dados pessoais sintéticos",
        "link": "n/a", "date": "2026-11", "accessed": "2026-11-11", "structure": "ficha", "is_test": True,
        "volatile": "n/a",
    },
}
EXPECTED_FILES = list(DOCS)


def download_inputs():
    """Baixa os documentos do Drive; o que faltar vem das cópias do GitHub."""
    missing = [name for name in EXPECTED_FILES if not (INPUT_DIR / name).exists()]
    if not missing:
        return "arquivos já presentes"
    source = "Drive"
    try:
        import gdown
        staging = WORK_DIR / "drive_tmp"
        staging.mkdir(exist_ok=True)
        gdown.download_folder(url=DRIVE_FOLDER_URL, output=str(staging), quiet=True, use_cookies=False)
        for found in staging.rglob("*"):
            if found.is_file() and found.name in missing:
                shutil.copy(found, INPUT_DIR / found.name)
    except Exception as error:  # noqa: BLE001 - o Drive pode limitar o acesso; seguimos para a reserva
        print("Aviso: falha no Drive:", type(error).__name__, str(error)[:120])
    still_missing = [name for name in EXPECTED_FILES if not (INPUT_DIR / name).exists()]
    if still_missing:
        source = "Drive + GitHub"
        for name in still_missing:
            urllib.request.urlretrieve(GITHUB_RAW_BASE + name, INPUT_DIR / name)
    final_missing = [name for name in EXPECTED_FILES if not (INPUT_DIR / name).exists()]
    if final_missing:
        raise FileNotFoundError(f"Documentos ausentes: {final_missing}")
    return f"baixados de: {source}"


print(download_inputs())
print(f"\n{'arquivo':34} {'formato':6} {'data':8} órgão")
for name, meta in DOCS.items():
    size_kb = (INPUT_DIR / name).stat().st_size / 1024
    print(f"{name:34} {Path(name).suffix[1:]:6} {meta['date']:8} {meta['agency'][:45]}  ({size_kb:.0f} KB)")

baixados de: Drive

arquivo                            formato data     órgão
cdc_lei_8078_1990.html             html   1990-09  Presidência da República  (169 KB)
lei_14181_2021.html                html   2021-07  Presidência da República  (51 KB)
decreto_13135_2026.docx            docx   2026-10  Presidência da República  (40 KB)
decreto_7962_2013.txt              txt    2013-03  Presidência da República  (6 KB)
guia_senacon_black_friday.pdf      pdf    2024-11  Ministério da Justiça e Segurança Pública / S  (365 KB)
documento_teste_ficticio.txt       txt    2026-11  Fictício — dados pessoais sintéticos  (1 KB)


### Upload manual e texto colado

O notebook também aceita arquivos enviados pelo usuário (PDF, TXT, DOCX e HTML) e **texto colado**.
As duas funções abaixo registram o documento com seus metadados; na execução gravada usamos o
texto colado abaixo apenas como demonstração (ele é fictício e também fica fora dos corpus).

In [4]:
def upload_documents():
    """Abre o seletor de arquivos do Colab e registra cada arquivo enviado (uso interativo)."""
    from google.colab import files  # disponível só no Colab
    uploaded = files.upload()
    for name, content in uploaded.items():
        (INPUT_DIR / name).write_bytes(content)
        DOCS.setdefault(name, {
            "title": name, "agency": "informar", "link": "informar", "date": "informar",
            "accessed": time.strftime("%Y-%m-%d"), "structure": "legal", "is_test": False, "volatile": "informar",
        })
    return list(uploaded)


def ingest_pasted_text(name, text, meta):
    """Registra um texto colado como se fosse um arquivo .txt."""
    (INPUT_DIR / name).write_text(text, encoding="utf-8")
    DOCS[name] = meta
    return name


PASTED_DEMO = (
    "Protocolo de teste 0001. A consumidora Joana Prado Mendes, CPF 111.444.777-35, pediu retorno pelo "
    "telefone (21) 99876-5432 ou pelo e-mail joana.prado@exemplo.org sobre uma cobrança que ela não "
    "reconhece. O Procon-SP foi informado."
)
ingest_pasted_text(
    "texto_colado_demo.txt", PASTED_DEMO,
    {"title": "Texto colado fictício (demonstração)", "agency": "Fictício", "link": "n/a",
     "date": "2026-11", "accessed": "2026-11-11", "structure": "ficha", "is_test": True, "volatile": "n/a"},
)
print("Documentos registrados:", len(DOCS), "| de teste (fora dos corpus):",
      [name for name, meta in DOCS.items() if meta["is_test"]])

Documentos registrados: 7 | de teste (fora dos corpus): ['documento_teste_ficticio.txt', 'texto_colado_demo.txt']


## 2. Extração de texto

Cada formato tem o seu leitor. A extração devolve **linhas brutas** com a origem (arquivo e
página, quando existe). Em HTML, cada parágrafo (`<p>`) vira uma linha e os trechos riscados
(dispositivos revogados, marcados com `<s>`) são descartados já na leitura. Em PDF, TXT e DOCX
as linhas vêm como no arquivo e são reunidas em parágrafos na etapa de limpeza.

In [5]:
def extract_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    lines = []
    for page_number, page in enumerate(reader.pages, start=1):
        for line in (page.extract_text() or "").splitlines():
            lines.append({"text": line, "page": page_number})
    return lines, False


def extract_html(path):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(path.read_bytes(), "lxml")        # detecta a codificação (windows-1252 no Planalto)
    for tag in soup(["script", "style", "s", "strike", "del"]):
        tag.decompose()                                    # remove também o texto riscado (revogado)
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all(["p", "h1", "h2", "h3", "li"])]
    if not paragraphs:
        paragraphs = soup.get_text("\n").splitlines()
    return [{"text": text, "page": None} for text in paragraphs if text.strip()], True


def extract_docx(path):
    from docx import Document
    document = Document(str(path))
    return [{"text": p.text, "page": None} for p in document.paragraphs if p.text.strip()], False


def read_text_file(path):
    raw = path.read_bytes()
    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def extract_txt(path):
    return [{"text": line, "page": None} for line in read_text_file(path).splitlines() if line.strip()], False


EXTRACTORS = {".pdf": extract_pdf, ".html": extract_html, ".htm": extract_html,
              ".docx": extract_docx, ".txt": extract_txt}

RAW_DOCS = {}
for name, meta in DOCS.items():
    path = INPUT_DIR / name
    lines, whole_paragraphs = EXTRACTORS[path.suffix.lower()](path)
    RAW_DOCS[name] = {"lines": lines, "whole_paragraphs": whole_paragraphs}
    words = sum(len(item["text"].split()) for item in lines)
    print(f"{name:34} {len(lines):5} linhas brutas  {words:6} palavras")
total_words = sum(len(item["text"].split()) for name, doc in RAW_DOCS.items()
                  if not DOCS[name]["is_test"] for item in doc["lines"])
print(f"\nTotal dos documentos oficiais (bruto): {total_words} palavras "
      f"(mínimo do enunciado: cerca de 15 mil)")

cdc_lei_8078_1990.html               548 linhas brutas   13128 palavras
lei_14181_2021.html                  104 linhas brutas    2716 palavras
decreto_13135_2026.docx              388 linhas brutas    2677 palavras
decreto_7962_2013.txt                 80 linhas brutas     834 palavras
guia_senacon_black_friday.pdf        184 linhas brutas    1200 palavras
documento_teste_ficticio.txt          26 linhas brutas     215 palavras
texto_colado_demo.txt                  1 linhas brutas      32 palavras

Total dos documentos oficiais (bruto): 20555 palavras (mínimo do enunciado: cerca de 15 mil)


## 3. Limpeza

O que removemos e por quê:

| Ruído | Tratamento |
|---|---|
| Cabeçalho do Planalto (órgão, links "Vigência", "Mensagem de veto", "Regulamento") e preâmbulo "O PRESIDENTE DA REPÚBLICA… DECRETA" | removidos: não ensinam nada ao modelo |
| Rodapé (data, assinaturas de autoridades e "Este texto não substitui…") | removido: são assinaturas de atos, não conteúdo |
| Notas de alteração "(Redação dada…)", "(Incluído…)", "(Vide…)", "(NR)" | removidas: referências internas que não fazem parte do texto da norma |
| Dispositivos vetados ou revogados e trechos riscados | removidos: não estão em vigor |
| Números de página, linhas repetidas, URLs soltas, reticências de leis alteradoras e caracteres estranhos | removidos |
| Palavras quebradas por hífen no fim da linha ("descon-\nto") | **juntadas** ("desconto") |
| Linhas quebradas no meio da frase | reunidas em parágrafos |

**Não** removemos stop words nem fazemos stemming: um modelo de linguagem precisa do texto
completo e natural. As seções (título, capítulo, seção) e o número do artigo ficam registrados
como metadados de cada trecho.

In [6]:
CONTROL_CHARS_RE = re.compile(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f\u007f\u200b-\u200f\u2028\u2029\u2060\ufeff\ufffd]")
PRIVATE_USE_RE = re.compile(r"[\ue000-\uf8ff]")
BULLET_RE = re.compile(r"^[•·▪●\uf0b7\uf0a7‣]\s*")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.I)
PAGE_NUMBER_RE = re.compile(r"^\s*(?:-\s*)?(?:p[áa]gina\s+)?\d{1,3}(?:\s*(?:de|/)\s*\d{1,3})?(?:\s*-)?\s*$", re.I)
DOT_RUN_RE = re.compile(r"\.{4,}|…{2,}")
AMENDMENT_NOTE_RE = re.compile(
    r"\(\s*(?:Reda[çc][ãa]o dada|Inclu[íi]d[oa]|Renumerad[oa]|Revogad[oa]|Vide|Vig[êe]ncia|Regulament|"
    r"Veto|Vetad[oa]|Promulga[çc][ãa]o|Produ[çc][ãa]o de efeito|Mensagem de veto|Alterad[oa]|"
    r"Acrescentad[oa]|Suprimid[oa]|NR\b)[^)]*\)", re.I | re.S)
VOID_MARK_RE = re.compile(r"\(\s*(?:Vetad|Revogad)", re.I)
LABEL_ONLY_RE = re.compile(r"^[\"“”]?\s*(?:Art\.\s*\S+|§\s*\S+|[IVXLCDM]+\s*[-–—]|[a-z]\))?\s*[.\-–—\"“”\s]*$")
EMENTA_RE = re.compile(r"^(?:Dispõe|Altera|Regulamenta|Institui|Estabelece|Aprova)\b")
PREAMBLE_RE = re.compile(r"^(?:O PRESIDENTE|A PRESIDENTA)\b|^DECRETA:?$|\bDECRETA:?\s*$")
FOOTER_RE = re.compile(r"^Brasília,\s")
UNIT_START_RE = re.compile(
    r"^(?:Art\.|§|Parágrafo único|[IVXLCDM]{1,6}(?:\s*[-–—]|\s*$)|[a-z]\)|CAPÍTULO|TÍTULO|SEÇÃO|Seção|Capítulo|"
    r"Título|Subseção|SUBSEÇÃO|Dispõe|Regulamenta|Altera|O PRESIDENTE|A PRESIDENTA|DECRETA|Brasília|DECRETO|LEI\b)")
HEADING_RE = re.compile(
    r"^(TÍTULO|CAPÍTULO|SEÇÃO|SUBSEÇÃO)\s+([IVXLCDM]+|ÚNICO|ÚNICA)\b\s*[-–—]?\s*(.*)$", re.I)
ARTICLE_RE = re.compile(r"^[\"“”]?Art\.\s*(\d+)\s*[º°o]?\.?(?:\s*-\s*([A-Z]))?")
HEADING_LEVEL = {"TÍTULO": 1, "CAPÍTULO": 2, "SEÇÃO": 3, "SUBSEÇÃO": 4}
HEADING_ABBREV = {"TÍTULO": "Tít.", "CAPÍTULO": "Cap.", "SEÇÃO": "Seç.", "SUBSEÇÃO": "Subseç."}


def join_lines(previous, line):
    """Junta duas linhas; se a anterior termina em hífen no meio de uma palavra, junta sem o hífen."""
    if previous.endswith("-") and len(previous) > 1 and previous[-2].isalpha() and line[:1].islower():
        return previous[:-1] + line
    return previous + " " + line


def is_heading_line(line):
    letters = [char for char in line if char.isalpha()]
    return (len(letters) >= 4 and line == line.upper() and len(line) <= 70
            and not line.rstrip().endswith((".", ";", ":", "?")))


def reflow_lines(items, structure, stats):
    """Reúne as linhas quebradas em parágrafos; cada parágrafo guarda a página da sua primeira linha."""
    paragraphs, current, current_page, current_is_heading = [], None, None, False
    for item in items:
        line = item["text"].strip()
        if not line:
            continue
        starts_new = False
        if current is None:
            starts_new = True
        elif structure == "legal":
            starts_new = bool(UNIT_START_RE.match(line))
        elif structure == "guia":
            heading = is_heading_line(line)
            bullet = bool(BULLET_RE.match(line))
            if heading and current_is_heading:
                starts_new = False
            elif heading or bullet or current_is_heading or item["page"] != current_page:
                starts_new = True
            elif current.endswith((".", ":", "?", "!", ")")) and line[:1].isupper():
                starts_new = True
        else:  # ficha: linhas de campos ficam separadas; continuação em minúscula é juntada
            starts_new = not (line[:1].islower() or current.endswith(","))
        if starts_new:
            if current is not None:
                paragraphs.append({"text": current, "page": current_page})
            current, current_page = line, item["page"]
            current_is_heading = structure == "guia" and is_heading_line(line)
        else:
            if current.endswith("-") and current[-2:-1].isalpha() and line[:1].islower():
                stats["palavras quebradas por hífen juntadas"] += 1
            current = join_lines(current, line)
    if current is not None:
        paragraphs.append({"text": current, "page": current_page})
    return paragraphs


def clean_paragraph(text, stats):
    """Limpeza de um parágrafo: notas de alteração, reticências, URLs, marcadores. Devolve '' se vazio."""
    text = BULLET_RE.sub("- ", text)
    text = CONTROL_CHARS_RE.sub("", text.replace("\u00ad", ""))
    text = PRIVATE_USE_RE.sub("", text)
    had_void = bool(VOID_MARK_RE.search(text))
    text, count = AMENDMENT_NOTE_RE.subn("", text)
    stats["notas de alteração removidas"] += count
    text, count = DOT_RUN_RE.subn(" ", text)
    stats["reticências removidas"] += count
    text, count = URL_RE.subn("", text)
    stats["URLs removidas"] += count
    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip("\"“”").strip()
    if had_void and LABEL_ONLY_RE.match(text):
        stats["dispositivos vetados/revogados removidos"] += 1
        return ""
    if not text or not re.search(r"[A-Za-zÀ-ÿ0-9]", text):
        return ""
    return text


def label_section(stack):
    """Monta o rótulo de seção, ex.: 'Cap. IV, Seç. II – Da Responsabilidade por Vício…'."""
    if not stack:
        return None
    levels = sorted(stack)
    labels = ", ".join(stack[level][0] for level in levels)
    title = stack[levels[-1]][1]
    return f"{labels} – {title}" if title else labels


def clean_document(name, raw):
    """Executa a limpeza de um documento e devolve (unidades, estatísticas).

    Cada unidade é um parágrafo limpo com origem (página), seção e artigo.
    """
    stats = Counter()
    structure = DOCS[name]["structure"]
    lines = []
    for item in raw["lines"]:
        text, removed = CONTROL_CHARS_RE.subn("", item["text"].replace("\u00ad", ""))
        text = PRIVATE_USE_RE.sub("", text) if not BULLET_RE.match(text) else text
        stats["caracteres estranhos removidos"] += removed
        lines.append({"text": text, "page": item["page"]})

    # Números de página e linhas repetidas (cabeçalhos e rodapés de PDF)
    kept = []
    for item in lines:
        if PAGE_NUMBER_RE.match(item["text"]):
            stats["números de página removidos"] += 1
            continue
        kept.append(item)
    # Cabeçalho/rodapé de PDF: linha que se repete na 1ª ou na última posição de pelo menos 3 páginas
    edge_pages = defaultdict(set)
    by_page = defaultdict(list)
    for item in kept:
        if item["page"] is not None:
            by_page[item["page"]].append(item["text"].strip().lower())
    for page_number, page_lines in by_page.items():
        for edge in (page_lines[0], page_lines[-1]):
            edge_pages[edge].add(page_number)
    repeated = {text for text, pages in edge_pages.items() if len(pages) >= 3 and len(text) <= 80}
    deduped, previous = [], None
    for item in kept:
        key = item["text"].strip().lower()
        if key in repeated or (previous is not None and key == previous and key and item["page"] is None):
            stats["linhas repetidas removidas"] += 1
            continue
        deduped.append(item)
        previous = key
    lines = deduped

    # Cabeçalho: tudo antes da ementa (documentos legais)
    if structure == "legal":
        for index, item in enumerate(lines):
            if EMENTA_RE.match(item["text"].strip()):
                stats["linhas de cabeçalho removidas"] += index
                lines = lines[index:]
                break

    # Linhas -> parágrafos
    if raw["whole_paragraphs"]:
        paragraphs = [{"text": item["text"].strip(), "page": item["page"]} for item in lines]
    else:
        paragraphs = reflow_lines(lines, structure, stats)

    units, stack, article, footer_reached = [], {}, None, False
    for item in paragraphs:
        paragraph = item["text"]
        if footer_reached:
            stats["rodapé/assinaturas removidos"] += 1
            continue
        if structure == "legal" and FOOTER_RE.match(paragraph):
            footer_reached = True
            stats["rodapé/assinaturas removidos"] += 1
            continue
        if structure == "legal" and PREAMBLE_RE.search(paragraph):
            stats["preâmbulo removido"] += 1
            continue
        text = clean_paragraph(paragraph, stats)
        if not text:
            continue
        # Títulos: guardam a seção e saem do texto
        if structure == "legal":
            match = HEADING_RE.match(text)
            if match:
                kind = match.group(1).upper()
                level = HEADING_LEVEL[kind]
                stack = {lvl: val for lvl, val in stack.items() if lvl < level}
                title = match.group(3).strip()
                title = title.capitalize() if title.isupper() else title
                stack[level] = (f"{HEADING_ABBREV[kind]} {match.group(2).upper()}", title)
                stats["títulos movidos para metadado"] += 1
                continue
            if stack and not ARTICLE_RE.match(text) and re.fullmatch(r"[A-ZÀ-Ú ,.\-–]+", text) and len(text) < 90:
                level = max(stack)
                stack[level] = (stack[level][0], text.capitalize() if not stack[level][1] else stack[level][1])
                stats["títulos movidos para metadado"] += 1
                continue
            art_match = ARTICLE_RE.match(text)
            if art_match:
                article = art_match.group(1) + (f"-{art_match.group(2)}" if art_match.group(2) else "")
        elif structure == "guia" and is_heading_line(text):
            stack = {1: ("", text.capitalize())}
            stats["títulos movidos para metadado"] += 1
            continue
        if re.match(r"^Este texto não substitui", text) or text == "*":
            stats["rodapé/assinaturas removidos"] += 1
            continue
        section = label_section(stack)
        if section and section.startswith(" – "):
            section = section[3:]
        units.append({"text": text, "section": section, "art": article if structure == "legal" else None,
                      "page": item["page"]})
    return units, stats


UNITS = {}
CLEAN_STATS = {}
for name in DOCS:
    UNITS[name], CLEAN_STATS[name] = clean_document(name, RAW_DOCS[name])

for name in DOCS:
    before = sum(len(item["text"].split()) for item in RAW_DOCS[name]["lines"])
    after = sum(len(unit["text"].split()) for unit in UNITS[name])
    print(f"\n{name}: {before} → {after} palavras, {len(UNITS[name])} parágrafos")
    for kind, count in CLEAN_STATS[name].most_common():
        print(f"    {kind}: {count}")


cdc_lei_8078_1990.html: 13128 → 11991 palavras, 465 parágrafos
    notas de alteração removidas: 138
    títulos movidos para metadado: 35
    dispositivos vetados/revogados removidos: 32
    linhas de cabeçalho removidas: 8
    rodapé/assinaturas removidos: 4
    linhas repetidas removidas: 1
    preâmbulo removido: 1
    caracteres estranhos removidos: 0
    reticências removidas: 0
    URLs removidas: 0

lei_14181_2021.html: 2716 → 2635 palavras, 84 parágrafos
    reticências removidas: 13
    notas de alteração removidas: 10
    rodapé/assinaturas removidos: 7
    linhas de cabeçalho removidas: 3
    dispositivos vetados/revogados removidos: 2
    caracteres estranhos removidos: 0
    URLs removidas: 0

decreto_13135_2026.docx: 2677 → 2580 palavras, 106 parágrafos
    linhas de cabeçalho removidas: 6
    títulos movidos para metadado: 6
    preâmbulo removido: 1
    rodapé/assinaturas removidos: 1
    caracteres estranhos removidos: 0
    notas de alteração removidas: 0
    reticê

### Exemplos antes e depois da limpeza

In [7]:
def show_before_after(title, before, after):
    print(f"--- {title}")
    print("ANTES :", before[:300])
    print("DEPOIS:", after[:300], "\n")


# 1) Nota de alteração (CDC, art. 57, §1º, alterado pela MP 1.393/2026)
sample = next((item["text"] for item in RAW_DOCS["cdc_lei_8078_1990.html"]["lines"]
               if "Medida Provisória nº 1.393" in item["text"] or "Medida Provis" in item["text"]), "")
if sample:
    show_before_after("Nota de alteração", sample, clean_paragraph(sample, Counter()))

# 2) Palavra quebrada por hífen e linhas repetidas (PDF)
pdf_lines = RAW_DOCS["guia_senacon_black_friday.pdf"]["lines"]
hyphen_index = next((i for i, item in enumerate(pdf_lines) if item["text"].rstrip().endswith("-")), None)
if hyphen_index is not None:
    pair = pdf_lines[hyphen_index:hyphen_index + 2]
    before = " / ".join(item["text"] for item in pair)
    after = reflow_lines(pair, "guia", Counter())[0]["text"]
    show_before_after("Palavra quebrada por hífen (PDF)", before, after)

# 3) Reticências e marcador (NR) de lei alteradora (Lei 14.181)
sample = next((item["text"] for item in RAW_DOCS["lei_14181_2021.html"]["lines"] if "(NR)" in item["text"]), "")
if sample:
    show_before_after("Reticências e (NR)", sample, clean_paragraph(sample, Counter()))

--- Palavra quebrada por hífen (PDF)
ANTES : baixos para garantir os presentes do fim do ano ou até mes- / mo conquistar algo que deseja muito.
DEPOIS: baixos para garantir os presentes do fim do ano ou até mesmo conquistar algo que deseja muito. 

--- Reticências e (NR)
ANTES : X - prevenção e tratamento do superendividamento como 
			forma de evitar a exclusão social do consumidor.” (NR)
DEPOIS: X - prevenção e tratamento do superendividamento como forma de evitar a exclusão social do consumidor. 



## 4. Normalização

Padronizamos codificação (UTF-8, com `ftfy` para corrigir textos quebrados), forma Unicode
**NFC**, espaços, quebras de linha e aspas. **Mantemos** acentos, pontuação e maiúsculas.
Também trocamos o grau (`°`) pelo ordinal (`º`), porque o Planalto mistura os dois
("§ 1°" e "§ 1º").

In [8]:
import ftfy

QUOTE_MAP = str.maketrans({"“": '"', "”": '"', "«": '"', "»": '"', "„": '"',
                           "‘": "'", "’": "'", "‚": "'", "´": "'", "`": "'"})
SPACES_RE = re.compile(r"[ \t\u00a0\u2000-\u200a\u202f\u205f\u3000]+")


def normalize_text(text):
    text = ftfy.fix_text(text, normalization="NFC")
    text = unicodedata.normalize("NFC", text).translate(QUOTE_MAP)
    text = re.sub(r"(?<=\d)\s?°", "º", text)
    text = SPACES_RE.sub(" ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)
    return text.strip()


changed = 0
for name in DOCS:
    for unit in UNITS[name]:
        normalized = normalize_text(unit["text"])
        changed += normalized != unit["text"]
        unit["text"] = normalized
print(f"Parágrafos alterados pela normalização: {changed}")
broken = "VocÃª nÃ£o precisa pagar â€œtaxaâ€\u009d nenhuma. O art. 8° garante 7 dias."
print("ANTES :", broken)
print("DEPOIS:", normalize_text(broken))

Parágrafos alterados pela normalização: 110
ANTES : VocÃª nÃ£o precisa pagar â€œtaxaâ€ nenhuma. O art. 8° garante 7 dias.
DEPOIS: Você não precisa pagar "taxa" nenhuma. O art. 8º garante 7 dias.


## 5. Anonimização (antes de gerar qualquer corpus)

Dados pessoais são trocados por um marcador do seu tipo — `[CPF]`, `[EMAIL]`, `[TELEFONE]`,
`[NOME]` — e **nenhuma tabela de correspondência é guardada**: sem como reverter, o dado está
anonimizado (e não apenas pseudonimizado). O notebook só conta quantos itens de cada tipo foram
mascarados.

| Tipo | Como detectamos |
|---|---|
| CPF | expressão regular; sem pontuação, só se os dígitos verificadores forem válidos |
| E-mail | expressão regular |
| Telefone | expressões regulares para celular e fixo, com ou sem DDD (intervalos de anos, como "2019-2022", são ignorados) |
| Nome de pessoa | NER do spaCy (`pt_core_news_lg`) para **nomes completos** (2 ou mais palavras), mais campos rotulados ("Nome:", "Atendente:") e tratamentos ("dona Maria", "Dra. Fernanda"). Palavras soltas não são mascaradas por NER, porque no texto legal ele erra muito ("II", "DAS", "Fui"); essa é uma limitação conhecida |

**Exceções (não mascaramos):**
- **Leis, códigos e normas:** Código de Defesa do Consumidor, Lei Maria da Penha, Lei 14.181, decretos etc.
- **Instituições e órgãos:** Senacon, Procon, Procon-SP, Ministério da Justiça e Segurança Pública, Anatel, Banco Central, Presidência da República, consumidor.gov.br.
- **Figuras públicas e autoridades** citadas em atos oficiais (as assinaturas do rodapé já saem na limpeza).
- **Pessoas jurídicas e marcas.**

In [9]:
import spacy

import importlib

importlib.invalidate_caches()
try:
    NLP = spacy.load(SPACY_MODEL)
except OSError:                      # o Colab às vezes só enxerga o modelo recém-instalado assim
    NLP = importlib.import_module(SPACY_MODEL).load()

INSTITUTION_TOKENS = {
    "senacon", "procon", "procons", "procon-sp", "anatel", "aneel", "anvisa", "anac", "ans", "bacen",
    "cade", "inmetro", "idec", "sebrae", "sndc", "cdc", "ufir", "dou", "oab", "fiocruz", "sus",
    "ministério", "ministerio", "justiça", "justica", "presidência", "presidencia", "república",
    "banco", "central", "defensoria", "juizado", "congresso", "senado", "câmara", "camara",
}
PUBLIC_FIGURES = {"luiz inácio lula da silva", "fernando collor", "jair messias bolsonaro"}
LAW_CONTEXT_RE = re.compile(
    r"(?:lei|código|codigo|estatuto|decreto|súmula|sumula|emenda|portaria|resolução|resolucao)\s*(?:n[º°o.]*\s*)?$", re.I)

EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
CPF_FORMATTED_RE = re.compile(r"(?<![\d.])\d{3}\.\d{3}\.\d{3}-\d{2}(?![\d-])")
CPF_LOOSE_RE = re.compile(r"(?<![\d.,/-])\d{3}[.\s]?\d{3}[.\s]?\d{3}[-\s]?\d{2}(?![\d/-])")
PHONE_RES = [
    re.compile(r"(?<!\d)(?:\+?55\s?)?(?:\(\d{2}\)|\d{2})\s?9\d{4}[-\s]?\d{4}(?!\d)"),          # celular com DDD
    re.compile(r"(?<!\d)(?:\+?55\s?)?(?:\(\d{2}\)\s?|\d{2}\s)[2-5]\d{3}[-\s]?\d{4}(?!\d)"),    # fixo com DDD
    re.compile(r"(?<![\d.,/])9?\d{4}-\d{4}(?![\d/])"),                                           # sem DDD
]
YEAR_PAIR_RE = re.compile(r"(?:19|20)\d{2}(?:19|20)\d{2}")
FIELD_NAME_RE = re.compile(
    r"(?im)^(\s*(?:nome(?: completo)?|consumidor(?:a)?|atendente|cliente|titular|respons[áa]vel)\s*:\s*)([^\n]+)$")
NAME_WORD = r"[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõçü'-]+"
TITLE_NAME_RE = re.compile(
    r"\b((?i:dona|sr\.?|sra\.?|srta\.?|dr\.?|dra\.?|atendente|senhor|senhora)\s+)"
    rf"({NAME_WORD}(?:(?:\s+(?:da|de|do|das|dos|e))?\s+{NAME_WORD}){{0,3}})")


def valid_cpf(digits):
    if len(digits) != 11 or digits == digits[0] * 11:
        return False
    for size in (9, 10):
        total = sum(int(digits[i]) * (size + 1 - i) for i in range(size))
        if (total * 10 % 11) % 10 != int(digits[size]):
            return False
    return True


def is_institution(text):
    words = re.findall(r"[\wçãõáéíóúâêô.-]+", text.lower())
    return any(word in INSTITUTION_TOKENS for word in words) or text.lower().strip() in PUBLIC_FIGURES


def regex_mask(text, counts):
    """Máscaras por expressão regular (e-mail, telefone, CPF) e por rótulo/tratamento (nomes)."""
    text, n = EMAIL_RE.subn("[EMAIL]", text)
    counts["EMAIL"] += n

    def phone_sub(match):
        digits = re.sub(r"\D", "", match.group(0))
        if YEAR_PAIR_RE.fullmatch(digits):
            return match.group(0)
        counts["TELEFONE"] += 1
        return "[TELEFONE]"

    for pattern in PHONE_RES[:2]:
        text = pattern.sub(phone_sub, text)

    def cpf_sub(match):
        counts["CPF"] += 1
        return "[CPF]"

    text = CPF_FORMATTED_RE.sub(cpf_sub, text)
    text = CPF_LOOSE_RE.sub(lambda m: cpf_sub(m) if valid_cpf(re.sub(r"\D", "", m.group(0))) else m.group(0), text)
    text = PHONE_RES[2].sub(phone_sub, text)

    def field_sub(match):
        if is_institution(match.group(2)):
            return match.group(0)
        counts["NOME"] += 1
        return match.group(1) + "[NOME]"

    text = FIELD_NAME_RE.sub(field_sub, text)

    def title_sub(match):
        if is_institution(match.group(2)):
            return match.group(0)
        counts["NOME"] += 1
        return match.group(1) + "[NOME]"

    return TITLE_NAME_RE.sub(title_sub, text)


NAME_CONNECTORS = {"da", "de", "do", "das", "dos", "e"}
MARKERS = {"CPF", "EMAIL", "TELEFONE", "NOME"}


ROMAN_RE = re.compile(r"^[IVXLCDM]+$")


def looks_like_person(entity_text):
    """Nome completo de pessoa: 2 ou mais palavras iniciadas por maiúscula (conectores como 'da' e 'de'
    são aceitos), sem dígitos nem '§' e sem numerais romanos. Palavras isoladas ("Fui", "II", "DAS") são
    falsos positivos comuns do NER em texto legal; nomes soltos só são mascarados por título ou rótulo."""
    words = entity_text.replace(".", ". ").split()
    names = [word for word in words if word.lower() not in NAME_CONNECTORS]
    if len(names) < 2 or len(words) > 7 or re.search(r"[\d§]", entity_text):
        return False
    if any(ROMAN_RE.match(word) for word in names):
        return False
    return words[0][:1].isupper() and all(word[:1].isupper() or word.lower() in NAME_CONNECTORS for word in words)


def keep_entity(text, entity):
    """True quando a entidade PER não deve ser mascarada: marcador já aplicado, trecho que não é nome
    de pessoa, nome de lei, órgão ou figura pública (exceções)."""
    if entity.text.strip("[] ") in MARKERS or text[max(0, entity.start_char - 1):entity.start_char] == "[":
        return True
    if not looks_like_person(entity.text) or is_institution(entity.text):
        return True
    return bool(LAW_CONTEXT_RE.search(text[max(0, entity.start_char - 25):entity.start_char]))


def anonymize_texts(texts):
    """Anonimiza uma lista de textos. Devolve (textos anonimizados, contagem por tipo, exceções mantidas)."""
    counts, kept = Counter(), Counter()
    masked = [regex_mask(text, counts) for text in texts]
    results = []
    for text, doc in zip(masked, NLP.pipe(masked, batch_size=64)):
        pieces, last = [], 0
        for entity in doc.ents:
            if entity.label_ != "PER" or "[NOME]" in entity.text:
                continue
            if keep_entity(text, entity):
                if is_institution(entity.text) or LAW_CONTEXT_RE.search(text[max(0, entity.start_char - 25):entity.start_char]):
                    kept[entity.text] += 1
                continue
            pieces.append(text[last:entity.start_char])
            pieces.append("[NOME]")
            last = entity.end_char
            counts["NOME"] += 1
        pieces.append(text[last:])
        results.append("".join(pieces))
    return results, counts, kept

### Demonstração: antes e depois (documento-teste fictício)

O documento abaixo é **inventado** e não entra nos corpus finais. Os CPFs, telefones, e-mails e
nomes são sintéticos.

In [10]:
ficha_text = read_text_file(INPUT_DIR / "documento_teste_ficticio.txt")
ficha_after, ficha_counts, ficha_kept = anonymize_texts([ficha_text])
print("=" * 30, "ANTES", "=" * 30)
print(ficha_text)
print("=" * 30, "DEPOIS", "=" * 29)
print(ficha_after[0])
print("\nItens mascarados por tipo:", {kind: total for kind, total in ficha_counts.items() if total})

must_be_gone = ["Maria Aparecida dos Santos Oliveira", "123.456.789-09", "987.654.321-00",
                "maria.oliveira1975@exemplo.com.br", "98765-4321", "3456-7890", "João Pedro Oliveira",
                "Carlos Eduardo Ferreira", "Fernanda Lima Costa", "dona Maria"]
must_stay = ["Procon-SP", "Senacon", "Código de Defesa do Consumidor", "Lei Maria da Penha", "consumidor.gov.br"]
leftover = [item for item in must_be_gone if item in ficha_after[0] and item != "dona Maria"]
lost = [item for item in must_stay if item not in ficha_after[0]]
print("Dados pessoais que sobraram:", leftover or "nenhum")
print("Exceções que foram mascaradas por engano:", lost or "nenhuma")
assert not leftover and not lost, "A anonimização do documento-teste falhou"

pasted_after, pasted_counts, _ = anonymize_texts([PASTED_DEMO])
print("\nTexto colado — antes :", PASTED_DEMO)
print("Texto colado — depois:", pasted_after[0])

============================== ANTES ==============================
FICHA DE ATENDIMENTO AO CONSUMIDOR Nº 2026/0457
DOCUMENTO FICTÍCIO PARA TESTE DE ANONIMIZAÇÃO. Todos os dados pessoais abaixo são inventados.

Data do registro: 05/11/2026
Canal: telefone

DADOS DO CONSUMIDOR
Nome: Maria Aparecida dos Santos Oliveira
CPF: 123.456.789-09
E-mail: maria.oliveira1975@exemplo.com.br
Telefone: (11) 98765-4321
Telefone alternativo: 11 3456-7890
Endereço: Rua das Acácias, 120, apto 34, São Paulo/SP

RELATO
A consumidora procurou o atendimento porque recebeu cobrança de um serviço de internet
que nunca contratou. Disse que o filho dela, João Pedro Oliveira, tentou cancelar pelo
SAC da operadora, mas foi transferido várias vezes e não resolveu. O atendente Carlos
Eduardo Ferreira registrou o protocolo 8890123 e orientou a consumidora a guardar as
faturas e a procurar o Procon-SP caso não houvesse resposta em 7 dias.

A consumidora também perguntou se o Código de Defesa do Consumidor garante o
ca

### Aplicação a todos os documentos

A anonimização roda sobre **todos** os parágrafos já limpos e normalizados, antes de qualquer
segmentação ou geração. Os documentos oficiais raramente trazem dados pessoais; o relatório abaixo
mostra o que foi mascarado e quais nomes foram preservados por exceção.

In [11]:
ANON_REPORT = {}
for name in DOCS:
    texts = [unit["text"] for unit in UNITS[name]]
    anonymized, counts, kept = anonymize_texts(texts)
    for unit, new_text in zip(UNITS[name], anonymized):
        unit["text"] = new_text
    masked_counts = {kind: total for kind, total in counts.items() if total}
    ANON_REPORT[name] = {"mascarados": masked_counts, "preservados": dict(kept)}
    print(f"{name:34} mascarados: {masked_counts or '-'}  | nomes preservados por exceção: {len(kept)}")

cdc_lei_8078_1990.html             mascarados: -  | nomes preservados por exceção: 0
lei_14181_2021.html                mascarados: -  | nomes preservados por exceção: 0
decreto_13135_2026.docx            mascarados: -  | nomes preservados por exceção: 0
decreto_7962_2013.txt              mascarados: -  | nomes preservados por exceção: 0
guia_senacon_black_friday.pdf      mascarados: -  | nomes preservados por exceção: 0
documento_teste_ficticio.txt       mascarados: {'EMAIL': 1, 'NOME': 6, 'CPF': 2, 'TELEFONE': 2}  | nomes preservados por exceção: 0
texto_colado_demo.txt              mascarados: {'EMAIL': 1, 'TELEFONE': 1, 'CPF': 1, 'NOME': 1}  | nomes preservados por exceção: 0


## 6. Segmentação em trechos de 256 a 512 tokens

Os trechos respeitam o fim de **parágrafos** (artigos, parágrafos, incisos) e, quando um parágrafo
não cabe, o fim de **frases**. Os tokens são contados com o tokenizador do
`Qwen/Qwen2.5-0.5B-Instruct`, o mesmo modelo da aula de fine-tuning. Cada trecho guarda de qual
arquivo, seção e artigo veio.

In [12]:
from transformers import AutoTokenizer

TOKENIZER = AutoTokenizer.from_pretrained(TOKENIZER_NAME)


def count_tokens(text):
    return len(TOKENIZER.encode(text, add_special_tokens=False))


SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?;])\s+(?=[A-ZÁÉÍÓÚÂÊÔÃÕÇ\"(§])")
ABBREVIATION_END_RE = re.compile(r"(?:\b[Aa]rts?|\bnº|\bn|\bDra?|\bSra?|\binc|\bpar)\.$")


def split_sentences(text):
    """Divide em frases sem quebrar abreviações como 'Art.' e 'nº'."""
    sentences = []
    for part in SENTENCE_SPLIT_RE.split(text):
        if sentences and ABBREVIATION_END_RE.search(sentences[-1]):
            sentences[-1] += " " + part
        else:
            sentences.append(part)
    return sentences


def split_long_text(text, limit):
    """Divide um texto maior que o limite em pedaços com até `limit` tokens, preferindo fins de frase."""
    pieces, current = [], ""
    for sentence in split_sentences(text):
        candidate = (current + " " + sentence).strip()
        if count_tokens(candidate) <= limit:
            current = candidate
            continue
        if current:
            pieces.append(current)
        if count_tokens(sentence) <= limit:
            current = sentence
        else:                                   # frase única maior que o limite: divide por palavras
            current = ""
            for word in sentence.split():
                candidate = (current + " " + word).strip()
                if count_tokens(candidate) > limit and current:
                    pieces.append(current)
                    current = word
                else:
                    current = candidate
    if current:
        pieces.append(current)
    return pieces


def chunk_label(segments, doc_title):
    base = segments[0]["section"] or doc_title
    articles = [segment["art"] for segment in segments if segment["art"]]
    if not articles:
        return base
    first, last = articles[0], articles[-1]
    return f"{base} (art. {first})" if first == last else f"{base} (arts. {first} a {last})"


def build_chunks(name):
    meta = DOCS[name]
    queue = []
    for unit in UNITS[name]:
        tokens = count_tokens(unit["text"])
        if tokens > MAX_TOKENS:
            for piece in split_long_text(unit["text"], MAX_TOKENS):
                queue.append({**unit, "text": piece, "tokens": count_tokens(piece)})
        else:
            queue.append({**unit, "tokens": tokens})

    chunks, current, current_tokens = [], [], 0

    def close():
        nonlocal current, current_tokens
        if current:
            chunks.append(current)
        current, current_tokens = [], 0

    while queue:
        piece = queue.pop(0)
        cost = piece["tokens"] + (1 if current else 0)
        if current_tokens + cost <= MAX_TOKENS:
            current.append(piece)
            current_tokens += cost
        elif current_tokens >= MIN_TOKENS:
            close()
            queue.insert(0, piece)
        else:                                    # trecho curto e o próximo parágrafo não cabe: preenche por frases
            sentences = split_sentences(piece["text"])
            taken, rest = [], list(sentences)
            while rest and current_tokens + count_tokens(" ".join(taken + rest[:1])) + 1 <= MAX_TOKENS:
                taken.append(rest.pop(0))
            if taken and rest:
                head = " ".join(taken)
                current.append({**piece, "text": head, "tokens": count_tokens(head)})
                close()
                tail = " ".join(rest)
                queue.insert(0, {**piece, "text": tail, "tokens": count_tokens(tail)})
            else:
                close()
                queue.insert(0, piece)
    close()

    # Último trecho curto: tenta juntar ao anterior; se não couber, puxa parágrafos do trecho anterior
    def chunk_size(segments):
        return count_tokens("\n".join(segment["text"] for segment in segments))

    if len(chunks) > 1 and chunk_size(chunks[-1]) < MIN_TOKENS:
        if chunk_size(chunks[-2] + chunks[-1]) <= MAX_TOKENS:
            chunks[-2] = chunks[-2] + chunks.pop()
        else:
            previous, last = list(chunks[-2]), list(chunks[-1])
            while len(previous) > 1 and chunk_size(last) < MIN_TOKENS:
                last.insert(0, previous.pop())
            if chunk_size(previous) >= MIN_TOKENS and chunk_size(last) <= MAX_TOKENS:
                chunks[-2], chunks[-1] = previous, last

    result = []
    for segments in chunks:
        text = "\n".join(segment["text"] for segment in segments)
        result.append({
            "text": text, "tokens": count_tokens(text), "fonte": name,
            "secao": chunk_label(segments, meta["title"]), "data": meta["date"],
            "titulo": meta["title"], "orgao": meta["agency"], "link": meta["link"],
        })
    return result


CHUNKS = {name: build_chunks(name) for name in DOCS if not DOCS[name]["is_test"]}

print(f"{'arquivo':34} {'trechos':>7} {'média':>6} {'mín':>5} {'máx':>5}  fora de 256–512")
for name, chunks in CHUNKS.items():
    sizes = [chunk["tokens"] for chunk in chunks]
    outside = sum(not (MIN_TOKENS <= size <= MAX_TOKENS) for size in sizes)
    print(f"{name:34} {len(chunks):7} {sum(sizes) / len(sizes):6.0f} {min(sizes):5} {max(sizes):5}  {outside}")
total_chunks = sum(len(chunks) for chunks in CHUNKS.values())
print(f"\nTotal de trechos: {total_chunks} (mínimo do enunciado: 50)")
assert total_chunks >= 50, "Menos de 50 trechos: reveja a limpeza ou inclua mais documentos"

arquivo                            trechos  média   mín   máx  fora de 256–512
cdc_lei_8078_1990.html                  45    469   281   505  0
lei_14181_2021.html                     10    452   299   497  0
decreto_13135_2026.docx                 10    437   282   504  0
decreto_7962_2013.txt                    3    448   374   492  0
guia_senacon_black_friday.pdf            4    454   379   492  0

Total de trechos: 72 (mínimo do enunciado: 50)


In [13]:
example = CHUNKS["cdc_lei_8078_1990.html"][12]
print(f"fonte: {example['fonte']}\nseção: {example['secao']}\ndata: {example['data']}\ntokens: {example['tokens']}\n")
print(example["text"][:900])

fonte: cdc_lei_8078_1990.html
seção: Tít. I, Cap. V, Seç. III – Da Publicidade (arts. 36 a 39)
data: 1990-09
tokens: 462

Parágrafo único. O fornecedor, na publicidade de seus produtos ou serviços, manterá, em seu poder, para informação dos legítimos interessados, os dados fáticos, técnicos e científicos que dão sustentação à mensagem.
Art. 37. É proibida toda publicidade enganosa ou abusiva.
§ 1º É enganosa qualquer modalidade de informação ou comunicação de caráter publicitário, inteira ou parcialmente falsa, ou, por qualquer outro modo, mesmo por omissão, capaz de induzir em erro o consumidor a respeito da natureza, características, qualidade, quantidade, propriedades, origem, preço e quaisquer outros dados sobre produtos e serviços.
§ 2º É abusiva, dentre outras a publicidade discriminatória de qualquer natureza, a que incite à violência, explore o medo ou a superstição, se aproveite da deficiência de julgamento e experiência da criança, desrespeita valores ambientais, ou que seja 

## 7. Língua real e conjunto de teste (criado **antes** dos corpus de treino)

### 7.1 Como o cidadão escreve de verdade

Pelo menos 30% das perguntas do usuário (em perguntas e respostas, conversa e teste) devem imitar
a escrita real: erros de digitação, caixa alta, abreviações e informalidade. As **respostas do
assistente seguem sempre em português correto e acessível**. Duas funções cuidam disso:
`make_informal` "suja" uma pergunta com regras sorteadas (semente fixa) e `looks_informal` mede
quantas perguntas parecem informais.

In [14]:
SHORT_FORMS = [(r"\bvocê\b", "vc"), (r"\bvocês\b", "vcs"), (r"\bporque\b", "pq"), (r"\bpor que\b", "pq"),
               (r"\bpara\b", "pra"), (r"\btambém\b", "tbm"), (r"\bnão\b", "nao"), (r"\bo que\b", "oq"),
               (r"\bmuito\b", "mt"), (r"\bhoje\b", "hj"), (r"\bcomigo\b", "cmg"), (r"\bestá\b", "ta"),
               (r"\bque\b", "q")]
INFORMAL_WORDS = {"vc", "vcs", "pq", "pra", "tbm", "tb", "q", "oq", "nao", "mt", "mto", "hj", "cmg", "obg",
                  "msm", "blz", "kkk", "kkkk", "mano", "aff", "ta", "tô", "né", "ne", "naum", "vlw"}
UNACCENTED_WORDS = {"voce", "entao", "ate", "so", "ja", "cartao", "reclamacao", "devolucao", "informacao",
                    "numero", "ingles", "robo", "preco", "cobranca"}


def strip_accents(text):
    return "".join(ch for ch in unicodedata.normalize("NFD", text) if unicodedata.category(ch) != "Mn")


def swap_adjacent_letters(text, rng):
    words = text.split(" ")
    candidates = [i for i, word in enumerate(words) if len(word) >= 5 and word.isalpha()]
    for index in rng.sample(candidates, min(len(candidates), rng.choice([1, 2]))):
        word = words[index]
        position = rng.randrange(1, len(word) - 2)
        words[index] = word[:position] + word[position + 1] + word[position] + word[position + 2:]
    return " ".join(words)


def apply_short_forms(text):
    for pattern, replacement in SHORT_FORMS:
        text = re.sub(pattern, replacement, text, flags=re.I)
    return text


def change_punctuation(text, rng):
    if text.endswith("?"):
        return text[:-1] + rng.choice(["??", "???", "", " ?"])
    return text.rstrip(".!") + rng.choice(["!!", "", "..."])


def add_interjection(text, rng):
    return rng.choice(["mano, ", "aff, ", "oi, "]) + text if rng.random() < 0.5 else text + rng.choice([" kkk", " por favor", " help"])


INFORMAL_RULES = {
    "upper": lambda text, rng: text.upper(),
    "no_accent": lambda text, rng: strip_accents(text),
    "short": lambda text, rng: apply_short_forms(text),
    "typo": swap_adjacent_letters,
    "punct": change_punctuation,
    "interjection": add_interjection,
    "lower": lambda text, rng: text.lower(),
}


def make_informal(text, rng):
    """'Suja' uma pergunta aplicando de 1 a 3 regras sorteadas. Nunca mexe nos números."""
    result = text
    for rule in rng.sample(list(INFORMAL_RULES), rng.choice([1, 2, 2, 3])):
        result = INFORMAL_RULES[rule](result, rng)
    if result == text:
        result = text[:1].lower() + text[1:].rstrip("?.!")
    return result


def looks_informal(text):
    stripped = text.strip()
    words = re.findall(r"[\wÀ-ÿ']+", stripped.lower())
    signals = [
        len(stripped) >= 6 and stripped == stripped.upper() and any(ch.isalpha() for ch in stripped),
        "??" in stripped or "!!" in stripped,
        stripped[:1].islower(),
        any(word in INFORMAL_WORDS or word in UNACCENTED_WORDS for word in words),
        bool(re.search(r"\s[,.?!]", stripped)),
    ]
    return any(signals)


demo_rng = random.Random(SEED)
for question in ["Onde posso reclamar de uma cobrança que eu não reconheço?",
                 "Posso devolver o produto que comprei pela internet?"]:
    print("limpa :", question)
    print("suja  :", make_informal(question, demo_rng), "\n")

limpa : Onde posso reclamar de uma cobrança que eu não reconheço?
suja  : ONDE POSSO RECLAMAR DE UMA COBRANÇA QUE EU NÃO RECONHEÇO? 

limpa : Posso devolver o produto que comprei pela internet?
suja  : posso devolver o produto que comprei pela internet? 



### 7.2 Conjunto de teste (20 perguntas no mínimo; aqui, 24)

Perguntas escritas à mão, com a resposta esperada, o tipo (`direta`, `fora_do_tema`, `urgencia` ou
`pegadinha`) e o documento de origem (`null` nas perguntas fora do tema). Todas as respostas foram
**conferidas contra o texto dos documentos**. Atenção: os números **192 (SAMU)** e **193
(Bombeiros)** das perguntas de urgência **não estão nos documentos**: o encaminhamento de
emergência é comportamento ensinado à mão, como pede o enunciado.

Este conjunto é gravado agora e **não é alterado depois**: guardamos o *hash* do arquivo e o
conferimos no fim do notebook. As frases são diferentes das usadas no treino (a seção 11 verifica
que não há vazamento).

In [15]:
CDC = "cdc_lei_8078_1990.html"
GUIA = "guia_senacon_black_friday.pdf"
SAC = "decreto_13135_2026.docx"
SUPER = "lei_14181_2021.html"

TEST_SET = [
    # ---- diretas
    ("oq eu faço se o produto que comprei veio com defeito??",
     "O fornecedor tem 30 dias para resolver. Se não resolver, você pode pedir a troca do produto, o abatimento no preço ou a devolução do dinheiro.",
     "direta", GUIA),
    ("Em quantos dias posso desistir de uma compra feita pela internet?",
     "Em até 7 dias, contados do recebimento do produto ou da assinatura do contrato.",
     "direta", GUIA),
    ("a loja cobrou 2x a mesma coisa no meu cartao, vou receber de volta?",
     "Sim. Quando a cobrança é indevida, você tem direito de receber de volta o valor pago a mais, em dobro e com correção, salvo engano justificável da empresa.",
     "direta", CDC),
    ("Quanto tempo tenho para reclamar de defeito em um celular novo?",
     "Para produto durável, como o celular, o prazo para reclamar de defeito aparente é de 90 dias, contados da entrega.",
     "direta", CDC),
    ("O que é publicidade enganosa?",
     "É a propaganda que passa informação falsa, ou que esconde dados importantes, e leva você a erro sobre o produto, o preço ou as condições.",
     "direta", CDC),
    ("PODEM COLOCAR MEU NOME NO SPC SEM ME AVISAR?",
     "Não. A empresa deve avisar você por escrito antes de registrar seu nome em um cadastro de inadimplentes.",
     "direta", CDC),
    ("A loja pode me obrigar a levar outro produto junto com o que eu quero?",
     "Não. Condicionar a venda à compra de outro produto, a chamada venda casada, é prática abusiva proibida.",
     "direta", CDC),
    ("prazo pra empresa responder minha reclamacao no sac?",
     "Nas empresas de serviços regulados pelo governo federal e nas instituições financeiras, como bancos, o SAC deve responder reclamações não urgentes em até 7 dias corridos.",
     "direta", SAC),
    ("Fui cobrado por uma assinatura que nunca contratei. Qual o prazo para resolverem?",
     "Nas empresas que seguem o decreto do SAC, a empresa deve suspender a cobrança de imediato e concluir a solução em até 48 horas.",
     "direta", SAC),
    ("tô endividada, dá pra renegociar tudo de uma vez?",
     "Sim. A lei do superendividamento permite pedir a renegociação das dívidas com audiência de conciliação com os credores, preservando o mínimo para você viver.",
     "direta", SUPER),
    ("Onde posso reclamar de uma loja online que não entregou meu pedido?",
     "No portal consumidor.gov.br ou no Procon da sua cidade ou estado.",
     "direta", GUIA),
    ("o manual do produto veio todo em ingles, pode?",
     "Não. As informações sobre o produto devem ser claras e em língua portuguesa.",
     "direta", CDC),
    # ---- fora do tema
    ("Como eu declaro o imposto de renda?",
     "Eu só ajudo com direitos do consumidor. Para o imposto de renda, procure o site ou o atendimento da Receita Federal.",
     "fora_do_tema", None),
    ("qual vacina tomar contra dengue",
     "Eu só ajudo com direitos do consumidor. Para vacinas, procure uma Unidade Básica de Saúde.",
     "fora_do_tema", None),
    ("quem ganhou o jogo ontem??",
     "Eu só ajudo com dúvidas sobre direitos do consumidor. Posso ajudar com alguma compra ou cobrança?",
     "fora_do_tema", None),
    ("Me indica um bom filme para hoje?",
     "Não consigo ajudar com isso, mas posso tirar dúvidas sobre seus direitos como consumidor.",
     "fora_do_tema", None),
    # ---- urgência
    ("meu filho bebeu produto de limpeza e ta passando mal!!",
     "Ligue agora para o 192 (SAMU). Não espere. Depois do atendimento, guarde a embalagem e registre a reclamação no Procon e com o fabricante.",
     "urgencia", [CDC, GUIA]),
    ("Instalaram o fogão novo e estou sentindo cheiro forte de gás na cozinha.",
     "Saia do local, não acenda luz nem fogo, abra as janelas se for seguro e ligue para o 193 (Bombeiros). Depois, acione o Procon e a empresa.",
     "urgencia", [CDC, GUIA]),
    ("o ventilador que comprei deu choque na minha mae, ela caiu",
     "Desligue a energia se puder fazer isso com segurança e ligue já para o 192 (SAMU). Depois, guarde o produto e a nota e avise o Procon e o fabricante.",
     "urgencia", [CDC, GUIA]),
    # ---- pegadinhas
    ("Comprei na loja física e me arrependi. Tenho 7 dias para devolver, né?",
     "Não necessariamente. Os 7 dias valem para compras feitas fora da loja, como pela internet ou por telefone. Na loja física, a troca depende de defeito ou da política da loja.",
     "pegadinha", GUIA),
    ("A loja subiu o preço na semana passada e agora anuncia 70% de desconto. Eles estão certos?",
     "Não necessariamente. Aumentar o preço para simular desconto, a chamada \"metade do dobro\", é publicidade enganosa. Você pode denunciar no Procon ou no consumidor.gov.br.",
     "pegadinha", GUIA),
    ("Recebi um produto que não pedi. Se eu ficar com ele, tenho que pagar?",
     "Não. Produto enviado sem você pedir é tratado como amostra grátis, e você não precisa pagar.",
     "pegadinha", CDC),
    ("o sac pode ser so robo, ne? nao precisa ter gente",
     "Não. Pode haver menu eletrônico no começo, mas as empresas obrigadas pelo decreto do SAC devem ter atendimento por uma pessoa das 8h às 20h, todos os dias.",
     "pegadinha", SAC),
    ("Se eu abrir a embalagem dentro dos 7 dias, perco o direito de devolver?",
     "Não. O CDC não exige embalagem fechada. Dentro do prazo você pode desistir da compra online e receber o dinheiro de volta.",
     "pegadinha", CDC),
]

TEST_PATH = OUT_DIR / "corpus_teste.jsonl"


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


test_rows = [{"pergunta": q, "resposta_esperada": a, "tipo": t, "fonte": f} for q, a, t, f in TEST_SET]
type_counts = Counter(row["tipo"] for row in test_rows)
assert len(test_rows) >= 20 and type_counts["fora_do_tema"] >= 3 and type_counts["urgencia"] >= 2
assert all(row["fonte"] is None for row in test_rows if row["tipo"] == "fora_do_tema")
write_jsonl(TEST_PATH, test_rows)
TEST_SHA256 = hashlib.sha256(TEST_PATH.read_bytes()).hexdigest()
test_informal = sum(looks_informal(row["pergunta"]) for row in test_rows)
print(f"corpus_teste.jsonl gravado: {len(test_rows)} perguntas | por tipo: {dict(type_counts)}")
print(f"Perguntas com língua real: {test_informal}/{len(test_rows)} ({test_informal / len(test_rows):.0%})")
print("SHA-256 (congelado):", TEST_SHA256)

corpus_teste.jsonl gravado: 24 perguntas | por tipo: {'direta': 12, 'fora_do_tema': 4, 'urgencia': 3, 'pegadinha': 5}
Perguntas com língua real: 12/24 (50%)
SHA-256 (congelado): b68e5c371a349ae7d655ffc3e49c9625b731358a63c2311de761c57e3492829c


## 8. Corpus de texto puro (`corpus_texto.jsonl`)

Os trechos limpos, normalizados e anonimizados da etapa de segmentação, com os metadados
(`fonte`, `secao`, `data`) — mais `titulo`, `orgao` e `link`, úteis para citar a origem no RAG.
Este corpus ensina o vocabulário e o estilo do tema e é a **base pronta para o RAG**.

Removemos duplicatas exatas (depois da normalização) e quase-duplicatas (sobreposição de
sequências de 5 palavras acima de 85%).

In [16]:
def dedup_key(text):
    return hashlib.sha1(re.sub(r"\W+", " ", text.lower()).strip().encode("utf-8")).hexdigest()


def shingles(text, size=5):
    words = re.findall(r"\w+", text.lower())
    return {" ".join(words[i:i + size]) for i in range(max(1, len(words) - size + 1))}


def jaccard(first, second):
    return len(first & second) / max(1, len(first | second))


text_rows, seen_keys, kept_shingles = [], set(), []
removed = Counter()
for name, chunks in CHUNKS.items():
    for chunk in chunks:
        key = dedup_key(chunk["text"])
        if key in seen_keys:
            removed["exatas"] += 1
            continue
        chunk_shingles = shingles(chunk["text"])
        if any(jaccard(chunk_shingles, other) >= 0.85 for other in kept_shingles):
            removed["quase-duplicatas"] += 1
            continue
        seen_keys.add(key)
        kept_shingles.append(chunk_shingles)
        text_rows.append({"text": chunk["text"], "fonte": chunk["fonte"], "secao": chunk["secao"],
                          "data": chunk["data"], "titulo": chunk["titulo"], "orgao": chunk["orgao"],
                          "link": chunk["link"]})

TEXT_PATH = OUT_DIR / "corpus_texto.jsonl"
write_jsonl(TEXT_PATH, text_rows)
TEXT_DUPLICATES = dict(removed)
print(f"corpus_texto.jsonl: {len(text_rows)} trechos (mínimo: 50) | duplicatas removidas: {TEXT_DUPLICATES or 0}")
print("trechos por documento:", dict(Counter(row["fonte"] for row in text_rows)))
assert len(text_rows) >= 50
print("\nExemplo:", json.dumps({**text_rows[3], "text": text_rows[3]["text"][:200] + "…"}, ensure_ascii=False, indent=1))

corpus_texto.jsonl: 68 trechos (mínimo: 50) | duplicatas removidas: {'exatas': 3, 'quase-duplicatas': 1}
trechos por documento: {'cdc_lei_8078_1990.html': 45, 'lei_14181_2021.html': 6, 'decreto_13135_2026.docx': 10, 'decreto_7962_2013.txt': 3, 'guia_senacon_black_friday.pdf': 4}

Exemplo: {
 "text": "X - a adequada e eficaz prestação dos serviços públicos em geral.\nXI - a garantia de práticas de crédito responsável, de educação financeira e de prevenção e tratamento de situações de superendividame…",
 "fonte": "cdc_lei_8078_1990.html",
 "secao": "Tít. I, Cap. III – Dos Direitos Básicos do Consumidor (arts. 6 a 8)",
 "data": "1990-09",
 "titulo": "Lei nº 8.078/1990 — Código de Defesa do Consumidor (texto compilado)",
 "orgao": "Presidência da República",
 "link": "https://www.planalto.gov.br/ccivil_03/Leis/L8078compilado.htm"
}


## 9. Corpus de perguntas e respostas (`corpus_qa.jsonl`)

Formato `instruction` / `input` / `output` (Alpaca), mais o campo `trecho` (a passagem que sustenta
a resposta) e `fonte`. As perguntas são geradas por um **modelo aberto rodando no Colab**
(Qwen2.5-7B-Instruct em 4 bits, sem API paga) a partir de cada trecho do corpus de texto puro e passam
por vários filtros:

1. **Estrutura:** JSON válido, pergunta e resposta com tamanho adequado, sem referência a "o trecho".
2. **Trecho de apoio completo:** o trecho gravado é ampliado até o dispositivo inteiro (com os incisos),
   para que ele realmente sustente a resposta e permita conferir a fidelidade.
3. **Fidelidade automática:** todo número da resposta precisa estar no trecho de apoio, a resposta precisa
   reusar as palavras do trecho, e perguntas sem contexto ("esse decreto") ou em primeira pessoa saem.
   **Fatos voláteis** (prazos e valores dos documentos voláteis e o valor da multa em Ufir) saem do QA:
   esses fatos ficam para o RAG. Depois, três **juízes** (LLM como juiz) confirmam: só informação do trecho,
   resposta pertinente e português correto; só ficam os pares com três "SIM".
4. **Duplicatas:** exatas e quase-duplicatas (similaridade de cosseno entre embeddings).
5. **Sem vazamento do teste:** nenhuma pergunta próxima de uma pergunta do conjunto de teste.
6. **Equilíbrio:** nenhum documento passa de 55% do corpus.
7. **Língua real:** uma parte das perguntas é "suja" por regras (seção 7.1).

As saídas do modelo ficam salvas em arquivo (`geracao_cache/`), então o notebook roda do início ao
fim sem chave de API e, se a sessão cair, retoma de onde parou.

In [17]:
import numpy as np
import torch

GENERATOR = {"tokenizer": None, "model": None}


def load_generator():
    """Carrega o Qwen2.5-7B-Instruct em 4 bits (NF4). Exige GPU (T4 do Colab)."""
    if GENERATOR["model"] is not None:
        return
    if not torch.cuda.is_available():
        raise RuntimeError("Sem GPU e sem cache de geração. No Colab: Ambiente de execução > "
                           "Alterar tipo de ambiente de execução > T4 GPU.")
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                      bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    GENERATOR["tokenizer"] = AutoTokenizer.from_pretrained(GENERATOR_NAME, padding_side="left")
    GENERATOR["model"] = AutoModelForCausalLM.from_pretrained(
        GENERATOR_NAME, quantization_config=quantization, device_map="auto")
    GENERATOR["model"].eval()
    print("Modelo carregado:", GENERATOR_NAME)


def llm_chat(batch_messages, max_new_tokens):
    """Gera a resposta (decodificação gulosa) para uma lista de conversas."""
    load_generator()
    tokenizer, model = GENERATOR["tokenizer"], GENERATOR["model"]
    prompts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
               for messages in batch_messages]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.eos_token_id)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return [tokenizer.decode(sequence, skip_special_tokens=True).strip() for sequence in new_tokens]


def run_llm_jobs(task, jobs, max_new_tokens, batch_size=6):
    """Executa tarefas (id, conversa) com cache em arquivo. Só carrega o modelo se houver pendências."""
    cache_path = CACHE_DIR / f"{task}.jsonl"
    done = {row["id"]: row["output"] for row in read_jsonl(cache_path)} if cache_path.exists() else {}
    pending = [(job_id, messages) for job_id, messages in jobs if job_id not in done]
    pending.sort(key=lambda job: -sum(len(m["content"]) for m in job[1]))
    started = time.time()
    for start in range(0, len(pending), batch_size):
        batch = pending[start:start + batch_size]
        outputs = llm_chat([messages for _, messages in batch], max_new_tokens)
        with open(cache_path, "a", encoding="utf-8") as handle:
            for (job_id, _), output in zip(batch, outputs):
                handle.write(json.dumps({"id": job_id, "output": output}, ensure_ascii=False) + "\n")
                done[job_id] = output
        if (start // batch_size) % 5 == 0:
            print(f"[{task}] {min(start + batch_size, len(pending))}/{len(pending)} novas "
                  f"({time.time() - started:.0f}s)")
    return {job_id: done[job_id] for job_id, _ in jobs}

In [18]:
from sentence_transformers import SentenceTransformer

EMBEDDER = SentenceTransformer(EMBEDDING_NAME)


def embed(texts):
    return EMBEDDER.encode(list(texts), normalize_embeddings=True, batch_size=64, show_progress_bar=False)


def norm_key(text):
    return re.sub(r"\W+", " ", strip_accents(text.lower())).strip()


def parse_json_block(text, opener, closer):
    """Extrai o primeiro bloco JSON (lista ou objeto) de uma resposta do modelo."""
    start, end = text.find(opener), text.rfind(closer)
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


NUMBER_WORDS = {"dois": 2, "duas": 2, "três": 3, "tres": 3, "quatro": 4, "cinco": 5, "seis": 6,
                "sete": 7, "oito": 8, "nove": 9, "dez": 10, "onze": 11, "doze": 12, "quinze": 15, "vinte": 20,
                "trinta": 30, "quarenta": 40, "cinquenta": 50, "sessenta": 60, "setenta": 70, "oitenta": 80,
                "noventa": 90, "cem": 100, "cento": 100, "duzentas": 200, "duzentos": 200}
COMPOUND_VALUES = {**NUMBER_WORDS, "um": 1, "uma": 1}   # "um"/"uma" só contam dentro de compostos ("vinte e um")
COMPOUND_RE = re.compile(r"\b(cento|vinte|trinta|quarenta|cinquenta|sessenta|setenta|oitenta|noventa)"
                         r"\s+e\s+(um|uma|dois|duas|três|tres|quatro|cinco|seis|sete|oito|nove|dez|vinte|trinta|"
                         r"quarenta|cinquenta|sessenta|setenta|oitenta|noventa)\b", re.I)


def numbers_in(text):
    """Valores numéricos de um texto, em dígitos ('30') ou por extenso ('trinta', 'quarenta e oito')."""
    values = {int(digits) for digits in re.findall(r"\d+", text)}
    for match in COMPOUND_RE.finditer(text):
        values.add(COMPOUND_VALUES[match.group(1).lower()] + COMPOUND_VALUES[match.group(2).lower()])
    for word in re.findall(r"[a-zç]+", text.lower()):
        if word in NUMBER_WORDS:
            values.add(NUMBER_WORDS[word])
    return values


def best_support(chunk_text, answer):
    """Frase(s) do trecho com maior sobreposição de palavras com a resposta (usada quando o 'trecho_apoio' do modelo não é literal)."""
    answer_words = set(re.findall(r"\w{4,}", answer.lower()))
    scored = []
    for sentence in split_sentences(chunk_text.replace("\n", " ")):
        words = set(re.findall(r"\w{4,}", sentence.lower()))
        scored.append((len(words & answer_words) / (len(words) ** 0.5 + 1), sentence))
    scored.sort(reverse=True)
    return " ".join(sentence for _, sentence in scored[:2])


REFERENCE_RE = re.compile(r"\b(?:no trecho|do trecho|o trecho|texto acima|segundo o texto|de acordo com o texto)\b", re.I)

QA_SYSTEM = ("Você ajuda a criar dados de treino para um atendente de defesa do consumidor. "
             "Responda SOMENTE com JSON válido, sem comentários.")


def qa_messages(chunk, pairs, avoid=()):
    avoid_text = ("\nNão repita estas perguntas já feitas:\n- " + "\n- ".join(avoid)) if avoid else ""
    user = (
        f"Documento: {chunk['titulo']}\nSeção: {chunk['secao']}\n\nTrecho:\n\"\"\"\n{chunk['text']}\n\"\"\"\n\n"
        f"Crie {pairs} pares de pergunta e resposta sobre o trecho.\n"
        "Regras:\n"
        "1. A pergunta deve soar como um consumidor leigo falando, em português simples, sem citar número de artigo "
        "e sem a palavra 'trecho'.\n"
        "2. A resposta usa SOMENTE informações do trecho, em no máximo 3 frases curtas e linguagem simples. "
        "Não invente prazos, valores, números ou nomes. Se a regra vale só para certos fornecedores, diga para quem vale.\n"
        "3. 'trecho_apoio' é a frase do trecho (copiada literalmente) que sustenta a resposta.\n"
        "4. Se o trecho não tiver informação útil para o consumidor, responda [].\n"
        f"{avoid_text}\n"
        'Formato: [{"pergunta": "...", "resposta": "...", "trecho_apoio": "..."}]')
    return [{"role": "system", "content": QA_SYSTEM}, {"role": "user", "content": user}]


def judge_messages(chunk_text, question, answer):
    user = (f"Trecho:\n\"\"\"\n{chunk_text}\n\"\"\"\n\nPergunta: {question}\nResposta: {answer}\n\n"
            "A resposta está totalmente sustentada pelo trecho, sem inventar nenhuma informação? "
            "Responda apenas SIM ou NÃO.")
    return [{"role": "system", "content": "Você é um revisor rigoroso de fidelidade."},
            {"role": "user", "content": user}]


def is_yes(output):
    return strip_accents(output.strip().lower()).startswith("sim")


def validate_pair(pair, chunk_text):
    """Valida a estrutura de um par; devolve (par limpo, motivo da rejeição)."""
    if not isinstance(pair, dict):
        return None, "formato"
    question = str(pair.get("pergunta", "")).strip()
    answer = str(pair.get("resposta", "")).strip()
    if not (12 <= len(question) <= 220 and question.endswith("?")):
        return None, "pergunta"
    if not (20 <= len(answer) <= 600) or REFERENCE_RE.search(question + " " + answer):
        return None, "resposta"
    if not numbers_in(answer) <= numbers_in(chunk_text):
        return None, "número fora do trecho"
    support = str(pair.get("trecho_apoio", "")).strip()
    flat_chunk = re.sub(r"\s+", " ", chunk_text)
    if not support or re.sub(r"\s+", " ", support) not in flat_chunk:
        support = best_support(chunk_text, answer)
    return {"pergunta": question, "resposta": answer, "trecho": support}, None



# --- Refinamento dos candidatos: trecho completo, fatos voláteis e apoio no texto ---------------------
LIST_ITEM_RE = re.compile(r"^(?:[IVXLCDM]+\s*[-–—]|[a-z]\)|\d+\s*[-–—)])")
CONTEXTLESS_RE = re.compile(r"\b(?:esse|este|essa|esta|esses|estes|essas|estas)\s+(?:decreto|lei|guia|código|artigo|regulamento|norma)\b"
                            r"|\b(?:essas|estas)\s+(?:regras|normas)\b", re.I)
FIRST_PERSON_RE = re.compile(r"^\s*(?:eu|meu|minha|meus|minhas)\b", re.I)
VOLATILE_WORDS_RE = re.compile(r"\b(?:prazos?|horas?|dias?|semanas?|mes(?:es)?|anos?|multas?|vigor|revog\w*|reais|por cento)\b|R\$|%", re.I)
FEE_RE = re.compile(r"Ufir|Unidade Fiscal de Refer", re.I)
STOPWORDS = {"para", "com", "que", "uma", "por", "mais", "como", "mas", "dos", "das", "nos", "nas", "ele", "ela",
             "eles", "elas", "seu", "sua", "seus", "suas", "esse", "essa", "isso", "este", "esta", "pode", "podem",
             "deve", "devem", "ser", "ter", "tem", "nao", "sim", "tambem", "quando", "onde", "qual", "quais",
             "sobre", "entre", "ainda", "desde", "ate", "pelo", "pela", "pelos", "pelas", "voce", "voces"}
MIN_SUPPORT_RATIO = 0.40      # fração mínima das palavras da resposta que aparecem no trecho de apoio
MAX_SUPPORT_CHARS = 900
VOLATILE_DOCS = {name for name, meta in DOCS.items() if str(meta.get("volatile", "")).startswith("sim")}


def flat_text(text):
    return re.sub(r"\s+", " ", text).strip()


def expand_support(chunk_text, support):
    """Amplia o trecho de apoio até o dispositivo inteiro: a linha completa, a introdução quando é um inciso
    e os incisos seguintes quando termina em dois-pontos. Assim o trecho gravado sustenta a resposta."""
    lines = [flat_text(line) for line in chunk_text.split("\n") if line.strip()]
    head = flat_text(support)[:60]
    index = next((i for i, line in enumerate(lines) if head and head in line), None)
    if index is None:
        return flat_text(support)
    picked = [index]
    if LIST_ITEM_RE.match(lines[index]) and index > 0 and lines[index - 1].endswith(":"):
        picked.insert(0, index - 1)
    last = picked[-1]
    while lines[last].endswith(":") and last + 1 < len(lines) and len(picked) < 5 and LIST_ITEM_RE.match(lines[last + 1]):
        last += 1
        picked.append(last)
    while last + 1 < len(lines) and len(picked) < 5 and LIST_ITEM_RE.match(lines[last + 1]) and LIST_ITEM_RE.match(lines[last]) \
            and len(" ".join(lines[i] for i in picked)) < 400:
        last += 1
        picked.append(last)
    text = " ".join(lines[i] for i in picked)
    if len(text) > MAX_SUPPORT_CHARS:
        text = text[:MAX_SUPPORT_CHARS].rsplit(" ", 1)[0] + "…"
    return text


def word_stems(text):
    words = re.findall(r"[a-z]{4,}", strip_accents(text.lower()))
    return {word[:5] for word in words if word not in STOPWORDS}


def support_ratio(answer, support):
    answer_stems = word_stems(answer)
    return len(answer_stems & word_stems(support)) / max(1, len(answer_stems))


def refine_candidate(candidate):
    """Amplia o trecho de apoio e aplica os filtros determinísticos. Devolve (par, motivo da rejeição)."""
    support = expand_support(text_rows[candidate["chunk"]]["text"], candidate["trecho"])
    question, answer = candidate["pergunta"], candidate["resposta"]
    if CONTEXTLESS_RE.search(question):
        return None, "pergunta sem contexto"
    if FIRST_PERSON_RE.search(answer):
        return None, "resposta em primeira pessoa"
    if FEE_RE.search(answer + " " + support):
        return None, "valor de multa em Ufir (volátil)"
    if not numbers_in(answer) <= numbers_in(support):
        return None, "número fora do trecho de apoio"
    if candidate["fonte"] in VOLATILE_DOCS and (VOLATILE_WORDS_RE.search(answer) or numbers_in(answer)):
        return None, "prazo ou valor de documento volátil (fica no RAG)"
    if support_ratio(answer, support) < MIN_SUPPORT_RATIO:
        return None, "resposta pouco apoiada no trecho"
    return {**candidate, "trecho": support}, None


def strict_judge_messages(kind, support, question, answer):
    """Três juízes independentes: fidelidade ao trecho, resposta direta à pergunta e português correto."""
    if kind == "fid":
        task = ("A resposta usa SOMENTE informações escritas no trecho? Se ela acrescenta qualquer explicação, exceção, "
                "prazo, valor, motivo ou consequência que não está no trecho, responda NÃO. Responda apenas SIM ou NÃO.")
    elif kind == "rel":
        task = ("A resposta responde diretamente ao que a pergunta quer saber, sem trocar de assunto nem inverter o "
                "sentido da pergunta? Responda apenas SIM ou NÃO.")
    else:
        task = ("A resposta está em português correto, com boa concordância e sem erros de digitação, e em linguagem "
                "simples? Responda apenas SIM ou NÃO.")
    user = f"Trecho:\n\"\"\"\n{support}\n\"\"\"\n\nPergunta: {question}\nResposta: {answer}\n\n{task}"
    return [{"role": "system", "content": "Você é um revisor rigoroso de dados de treino."}, {"role": "user", "content": user}]

### 9.1 Geração dos candidatos e filtros

In [19]:
PAIRS_PER_CHUNK = 4
QA_STATS = Counter()


def short_hash(*parts):
    return hashlib.sha1("|".join(str(part) for part in parts).encode("utf-8")).hexdigest()[:16]


def generate_qa_candidates(task, avoid_by_chunk=None):
    ids = [short_hash(chunk["text"], PAIRS_PER_CHUNK) for chunk in text_rows]
    jobs = [(job_id, qa_messages(chunk, PAIRS_PER_CHUNK, (avoid_by_chunk or {}).get(index, ())))
            for index, (job_id, chunk) in enumerate(zip(ids, text_rows))]
    outputs = run_llm_jobs(task, jobs, max_new_tokens=800, batch_size=6)
    candidates = []
    for index, chunk in enumerate(text_rows):
        pairs = parse_json_block(outputs[ids[index]], "[", "]") or []
        QA_STATS["pares gerados"] += len(pairs)
        for pair in pairs:
            cleaned, reason = validate_pair(pair, chunk["text"])
            if cleaned is None:
                QA_STATS[f"rejeitados: {reason}"] += 1
                continue
            candidates.append({**cleaned, "chunk": index, "fonte": chunk["fonte"]})
    return candidates


JUDGE_LABELS = {"fid": "juiz: informação fora do trecho", "rel": "juiz: não responde à pergunta",
                "lang": "juiz: português incorreto"}


def refine_and_judge(task, candidates):
    """Refina (trecho completo + filtros determinísticos) e passa pelos três juízes; só fica quem os três aprovam."""
    refined = []
    for candidate in candidates:
        cleaned, reason = refine_candidate(candidate)
        if cleaned is None:
            QA_STATS[f"rejeitados: {reason}"] += 1
        else:
            refined.append(cleaned)
    verdicts = {}
    for kind in JUDGE_LABELS:
        ids = [short_hash("v2", kind, c["chunk"], c["pergunta"], c["resposta"]) for c in refined]
        jobs = [(job_id, strict_judge_messages(kind, c["trecho"], c["pergunta"], c["resposta"]))
                for job_id, c in zip(ids, refined)]
        outputs = run_llm_jobs(f"{task}_{kind}", jobs, max_new_tokens=4, batch_size=8)
        verdicts[kind] = [is_yes(outputs[job_id]) for job_id in ids]
    approved = []
    for position, candidate in enumerate(refined):
        failed = [kind for kind in JUDGE_LABELS if not verdicts[kind][position]]
        if failed:
            QA_STATS[f"rejeitados: {JUDGE_LABELS[failed[0]]}"] += 1
        else:
            approved.append(candidate)
    return approved


test_embeddings = embed([row["pergunta"] for row in test_rows])
test_keys = {norm_key(row["pergunta"]) for row in test_rows}


def filter_duplicates_and_leaks(candidates):
    """Remove duplicatas exatas, quase-duplicatas e perguntas parecidas com as do teste."""
    kept, kept_vectors, seen = [], [], set()
    vectors = embed([c["pergunta"] for c in candidates]) if candidates else np.zeros((0, 1))
    for candidate, vector in zip(candidates, vectors):
        key = norm_key(candidate["pergunta"])
        if key in test_keys or float((test_embeddings @ vector).max()) >= LEAK_THRESHOLD:
            QA_STATS["removidos: parecidos com o teste"] += 1
            continue
        if key in seen:
            QA_STATS["duplicatas exatas removidas"] += 1
            continue
        if kept_vectors and float((np.array(kept_vectors) @ vector).max()) >= NEAR_DUP_THRESHOLD:
            QA_STATS["quase-duplicatas removidas"] += 1
            continue
        seen.add(key)
        kept.append(candidate)
        kept_vectors.append(vector)
    return kept


def select_balanced(candidates, target, max_share=0.55):
    """Escolhe até `target` pares: no máximo 2 por trecho na primeira volta e nenhum documento acima de `max_share`."""
    cap = int(target * max_share)
    by_chunk = defaultdict(list)
    for candidate in candidates:
        by_chunk[candidate["chunk"]].append(candidate)
    rng = random.Random(SEED)
    chunk_ids = sorted(by_chunk)
    rng.shuffle(chunk_ids)
    selected, per_doc = [], Counter()
    for round_number in range(PAIRS_PER_CHUNK):
        for chunk_id in chunk_ids:
            if len(selected) >= target:
                break
            if round_number < len(by_chunk[chunk_id]):
                candidate = by_chunk[chunk_id][round_number]
                if per_doc[candidate["fonte"]] < cap:
                    selected.append(candidate)
                    per_doc[candidate["fonte"]] += 1
    return selected


raw_candidates = generate_qa_candidates("qa_candidatos")
approved = refine_and_judge("qa_juiz_v2", raw_candidates)
pool = filter_duplicates_and_leaks(approved)
selected_qa = select_balanced(pool, TARGET_QA)

if len(selected_qa) < TARGET_QA:   # segunda passada, pedindo perguntas diferentes das já aceitas
    avoid = defaultdict(list)
    for candidate in pool:
        avoid[candidate["chunk"]].append(candidate["pergunta"])
    extra = refine_and_judge("qa_juiz_v2_2", generate_qa_candidates("qa_candidatos_2", avoid))
    for key in [k for k in QA_STATS if k.startswith(("duplicatas", "quase", "removidos"))]:
        del QA_STATS[key]                       # recontamos só a filtragem final
    pool = filter_duplicates_and_leaks(approved + extra)
    selected_qa = select_balanced(pool, TARGET_QA)

print(f"Candidatos válidos: {len(raw_candidates)} | aprovados pelo juiz: {len(approved)} | "
      f"após duplicatas e vazamento: {len(pool)} | selecionados: {len(selected_qa)}")
for reason, count in sorted(QA_STATS.items()):
    print(f"    {reason}: {count}")
assert len(selected_qa) >= 100, "Menos de 100 pares: aumente PAIRS_PER_CHUNK ou revise os filtros"

Modelo carregado: Qwen/Qwen2.5-7B-Instruct
[qa_candidatos] 6/68 novas (455s)
[qa_candidatos] 36/68 novas (1189s)
[qa_candidatos] 66/68 novas (1910s)
[qa_juiz_v2_fid] 8/171 novas (3s)
[qa_juiz_v2_fid] 48/171 novas (14s)
[qa_juiz_v2_fid] 88/171 novas (25s)
[qa_juiz_v2_fid] 128/171 novas (34s)
[qa_juiz_v2_fid] 168/171 novas (42s)
[qa_juiz_v2_rel] 8/171 novas (3s)
[qa_juiz_v2_rel] 48/171 novas (14s)
[qa_juiz_v2_rel] 88/171 novas (23s)
[qa_juiz_v2_rel] 128/171 novas (32s)
[qa_juiz_v2_rel] 168/171 novas (39s)
[qa_juiz_v2_lang] 8/171 novas (3s)
[qa_juiz_v2_lang] 48/171 novas (14s)
[qa_juiz_v2_lang] 88/171 novas (23s)
[qa_juiz_v2_lang] 128/171 novas (32s)
[qa_juiz_v2_lang] 168/171 novas (39s)
[qa_candidatos_2] 6/68 novas (139s)
[qa_candidatos_2] 36/68 novas (862s)
[qa_candidatos_2] 66/68 novas (1580s)
[qa_juiz_v2_2_fid] 8/146 novas (2s)
[qa_juiz_v2_2_fid] 48/146 novas (13s)
[qa_juiz_v2_2_fid] 88/146 novas (23s)
[qa_juiz_v2_2_fid] 128/146 novas (31s)
[qa_juiz_v2_2_rel] 8/146 novas (2s)
[qa_juiz

### 9.2 Língua real e gravação

Uma fração (`INFORMAL_RATE`) das perguntas é "suja" com as regras da seção 7.1. A resposta, o
`trecho` e a `fonte` não mudam. Depois de sujar, conferimos de novo que nenhuma pergunta ficou
parecida com as do teste.

In [20]:
rng_informal = random.Random(SEED + 1)
qa_rows = []
for candidate in selected_qa:
    question = candidate["pergunta"]
    if rng_informal.random() < INFORMAL_RATE:
        question = make_informal(question, rng_informal)
    qa_rows.append({"instruction": question, "input": "", "output": candidate["resposta"],
                    "trecho": candidate["trecho"], "fonte": candidate["fonte"]})

leak_scores = (embed([row["instruction"] for row in qa_rows]) @ test_embeddings.T).max(axis=1)
qa_rows = [row for row, score in zip(qa_rows, leak_scores) if score < LEAK_THRESHOLD]
QA_PATH = OUT_DIR / "corpus_qa.jsonl"
write_jsonl(QA_PATH, qa_rows)
qa_informal = sum(looks_informal(row["instruction"]) for row in qa_rows)
print(f"corpus_qa.jsonl: {len(qa_rows)} pares | documentos: {dict(Counter(row['fonte'] for row in qa_rows))}")
print(f"Perguntas com língua real: {qa_informal}/{len(qa_rows)} ({qa_informal / len(qa_rows):.0%})")
print("\nExemplo:", json.dumps(qa_rows[0], ensure_ascii=False, indent=1)[:900])

corpus_qa.jsonl: 126 pares | documentos: {'lei_14181_2021.html': 20, 'cdc_lei_8078_1990.html': 71, 'decreto_7962_2013.txt': 8, 'guia_senacon_black_friday.pdf': 10, 'decreto_13135_2026.docx': 17}
Perguntas com língua real: 48/126 (38%)

Exemplo: {
 "instruction": "No plano de pagamento, o juiz pode diminiur os encargos da dívida?",
 "input": "",
 "output": "Sim, podem ser incluídas medidas para reduzir os encargos da dívida.",
 "trecho": "§ 4º Constarão do plano de pagamento referido no § 3º deste artigo: I - medidas de dilação dos prazos de pagamento e de redução dos encargos da dívida ou da remuneração do fornecedor, entre outras destinadas a facilitar o pagamento da dívida; II - referência à suspensão ou à extinção das ações judiciais em curso; III - data a partir da qual será providenciada a exclusão do consumidor de bancos de dados e de cadastros de inadimplentes;",
 "fonte": "lei_14181_2021.html"
}


## 10. Corpus conversacional (`corpus_chat.jsonl`)

Formato de mensagens com papéis (`system`, `user`, `assistant`). Todos os diálogos têm **2 ou mais
trocas**, e a segunda pergunta depende da primeira ("E onde eu reclamo?"), para ensinar o modelo a
usar o histórico da conversa.

| Grupo | Qtd. | Como é feito |
|---|---|---|
| Normais | ≥ 24 | 22 gerados pelo modelo a partir dos trechos (6 temas), com juiz de fidelidade e de dependência entre as perguntas, e 2 escritos à mão |
| Fora do tema | 4 | escritos à mão |
| Orientação individual (advogado, Defensoria, Juizado, médico) | 4 | escritos à mão |
| **Urgência** (192 SAMU e 193 Bombeiros) | 4 | escritos à mão: é o caso que **não pode falhar** |

O 199 (Defesa Civil) não entra porque não faz parte do tema do CDC. As recusas e os
encaminhamentos somam **12** (o mínimo é 10).

In [21]:
SYSTEM_PROMPT = (
    "Você é um atendente de defesa do consumidor. Responda em português simples e educado, com frases curtas. "
    "Explique os direitos do consumidor sem dar parecer jurídico sobre casos individuais. Se a pergunta for de "
    "outro assunto, diga que só ajuda com direitos do consumidor. Se houver risco à vida ou à saúde, mande ligar "
    "para o 192 (SAMU) ou 193 (Bombeiros) antes de qualquer outra coisa. Prazos e valores podem mudar: peça para "
    "confirmar no Procon ou no consumidor.gov.br. Se não tiver certeza, encaminhe ao Procon, à Defensoria Pública "
    "ou a um advogado.")

# (tipo, fonte, [(usuário, atendente), (usuário, atendente)])
MANUAL_DIALOGUES = [
    # ---------------- fora do tema
    ("fora_do_tema", None, [
        ("Como faço para pedir o auxílio do governo?",
         "Eu só ajudo com direitos do consumidor, então não consigo orientar sobre benefícios do governo. Para isso, procure o CRAS da sua cidade ou o site gov.br."),
        ("Entendi. E sobre compra de produto você ajuda?",
         "Ajudo sim. Me conte o que aconteceu com a sua compra.")]),
    ("fora_do_tema", None, [
        ("qual remédio é bom pra dor de cabeça?",
         "Não posso indicar remédios nem fazer diagnóstico. Para dor de cabeça, procure um médico ou um posto de saúde. Eu ajudo com direitos do consumidor."),
        ("ok, e se a farmácia vendeu um remédio vencido?",
         "Isso eu ajudo. Produto vencido não pode ser vendido. Se você passou mal, ligue para o 192. Depois, guarde a embalagem e a nota e reclame no Procon.")]),
    ("fora_do_tema", None, [
        ("Quanto vai ficar o dólar amanhã?",
         "Não consigo prever isso, e eu só ajudo com direitos do consumidor. Para cotações, consulte o site do Banco Central."),
        ("e se eu comprar em dólar num site de fora, tenho direitos?",
         "Em geral, o CDC protege compras online. Com sites de fora do país a proteção pode ser mais difícil. Guarde os comprovantes e procure o Procon ou o consumidor.gov.br.")]),
    ("fora_do_tema", None, [
        ("Me ajuda a escrever uma redação sobre meio ambiente?",
         "Não consigo ajudar com redações. Eu só ajudo com direitos do consumidor."),
        ("e sobre o direito de trocar um produto, pode explicar?",
         "Claro. Se o produto vier com defeito e a loja não resolver em 30 dias, você pode escolher entre a troca, a devolução do dinheiro ou o abatimento no preço.")]),
    # ---------------- orientação individual
    ("orientacao_individual", CDC, [
        ("A loja me vendeu uma geladeira com defeito. Se eu processar, vou ganhar?",
         "Não posso dizer como o juiz vai decidir cada caso. O que posso explicar é que, se o defeito não for resolvido em 30 dias, você pode pedir a troca, o dinheiro de volta ou o abatimento no preço."),
        ("E quem me ajuda a entrar com o processo?",
         "Procure a Defensoria Pública, se você não puder pagar um advogado, ou o Juizado Especial Cível da sua cidade. O Procon também orienta e registra a reclamação.")]),
    ("orientacao_individual", CDC, [
        ("Quanto de indenização eu posso pedir por ter ficado com o nome sujo por engano?",
         "Eu não consigo calcular um valor, porque isso depende do seu caso. Posso dizer que você tem direito de exigir a correção do cadastro e pode pedir indenização na Justiça."),
        ("e com quem eu resolvo isso?",
         "Comece pela empresa que registrou o seu nome e pelo Procon. Para pedir indenização, procure um advogado, a Defensoria Pública ou o Juizado Especial.")]),
    ("orientacao_individual", CDC, [
        ("Você pode olhar meu contrato de academia e dizer se ele é abusivo?",
         "Não consigo analisar o seu contrato. Posso explicar que cláusulas abusivas, que deixam o consumidor em desvantagem exagerada, são nulas, isto é, não valem."),
        ("e quem faz essa análise?",
         "Um advogado, a Defensoria Pública ou o Procon podem ler o contrato e orientar você. Leve uma cópia do contrato e os seus comprovantes.")]),
    ("orientacao_individual", CDC, [
        ("comi uma comida vencida e to com dor de barriga, é grave?",
         "Eu não posso fazer diagnóstico. Se tiver vômito forte, febre, desmaio ou muita dor, ligue para o 192 (SAMU). Se estiver mais leve, procure um médico ou uma Unidade Básica de Saúde."),
        ("e a loja que vendeu, o que eu faço?",
         "Guarde a embalagem e a nota fiscal e, quando estiver melhor, registre a reclamação no Procon ou no consumidor.gov.br. Vender produto vencido não é permitido.")]),
    # ---------------- urgência
    ("urgencia", [CDC, GUIA], [
        ("meu filho de 2 anos engoliu a pilha de um brinquedo!!",
         "Ligue agora para o 192 (SAMU) e siga as orientações do atendente. Não espere."),
        ("e o brinquedo, o que eu faço depois?",
         "Guarde o brinquedo e a embalagem. Depois que seu filho estiver em atendimento, registre a reclamação no Procon e avise o fabricante. Brinquedo perigoso não pode ser vendido.")]),
    ("urgencia", [CDC, GUIA], [
        ("o botijão de gás que entregaram tá vazando, dá pra ouvir o chiado",
         "Saia de perto agora, não acenda luz nem fósforo e não ligue aparelhos. Ligue para o 193 (Bombeiros). Se puder fazer com segurança, deixe as janelas abertas."),
        ("e depois, reclamo com quem?",
         "Guarde a nota e tire fotos do botijão. Depois, reclame na revendedora, no Procon e no consumidor.gov.br. O fornecedor responde pelos danos de um produto com defeito.")]),
    ("urgencia", [CDC, GUIA], [
        ("o carregador do celular esquentou muito e começou a pegar fogo",
         "Ligue para o 193 (Bombeiros) agora. Se puder fazer isso sem risco, desligue da tomada e se afaste. Não jogue água no aparelho ligado na tomada."),
        ("e o carregador, eu devolvo na loja?",
         "Guarde o que sobrou do carregador, fotos e a nota fiscal. Depois, procure o Procon e o fabricante, porque o produto não oferece a segurança que se espera.")]),
    ("urgencia", [CDC, GUIA], [
        ("comi um biscoito sem aviso de amendoim no rótulo e minha boca tá inchando e tá faltando ar",
         "Isso é uma emergência. Ligue agora para o 192 (SAMU). Não espere melhorar."),
        ("e a empresa do biscoito?",
         "Depois do atendimento, guarde a embalagem e a nota e registre a reclamação no Procon. O fabricante deve informar de forma clara os riscos do produto no rótulo.")]),
    # ---------------- normais escritos à mão
    ("normal_manual", GUIA, [
        ("comprei um tênis no site e não gostei, posso devolver?",
         "Pode. Em compras pela internet, você tem 7 dias para desistir, contados do recebimento do produto."),
        ("e o dinheiro, eu recebo tudo de volta?",
         "Sim. A loja deve devolver o valor pago de imediato, com correção. Guarde o comprovante e peça o cancelamento pelo mesmo canal da compra.")]),
    ("normal_manual", SAC, [
        ("ninguém atende no sac do meu banco",
         "Bancos devem ter atendimento por uma pessoa das 8h às 20h, todos os dias. Confirme as regras atuais no Procon, porque elas mudaram em novembro de 2026."),
        ("e se eu reclamar e ficarem sem responder?",
         "Em geral, a resposta deve vir em até 7 dias corridos. Passou disso, registre a reclamação no consumidor.gov.br.")]),
]
REFUSAL_KINDS = {"fora_do_tema", "orientacao_individual", "urgencia"}
print("Diálogos escritos à mão:", Counter(kind for kind, _, _ in MANUAL_DIALOGUES))

Diálogos escritos à mão: Counter({'fora_do_tema': 4, 'orientacao_individual': 4, 'urgencia': 4, 'normal_manual': 2})


### 10.1 Diálogos normais gerados pelo modelo

Para cada tema, escolhemos os trechos mais próximos (por embeddings) e pedimos um diálogo de 2
trocas. Quatro juízes confirmam que (a) as respostas são fiéis ao trecho, (b) cada resposta é pertinente,
(c) o português está correto e (d) a segunda pergunta depende da primeira. A segunda pergunta varia de estilo
(onde, prazo, prova, esclarecimento...) para não ensinar o modelo a repetir sempre a mesma frase.

In [22]:
TOPICS = [
    ("defeito e garantia", 6, "produto com defeito vício garantia troca reparo abatimento do preço devolução do dinheiro prazo de 30 dias para reclamar"),
    ("compra online e arrependimento", 4, "compra pela internet direito de arrependimento 7 dias desistir do contrato fora do estabelecimento devolução do valor"),
    ("cobrança indevida e cadastro de inadimplentes", 4, "cobrança indevida devolução em dobro cadastro de inadimplentes negativação comunicação por escrito constrangimento"),
    ("oferta, publicidade e Black Friday", 3, "publicidade enganosa oferta preço desconto informação clara venda casada práticas abusivas"),
    ("SAC", 2, "serviço de atendimento ao consumidor SAC prazo de resposta atendimento por pessoa cancelamento"),
    ("superendividamento", 3, "superendividamento renegociação de dívidas mínimo existencial conciliação crédito responsável"),
]


SECOND_TURN_STYLES = [
    "pergunta curta sobre ONDE ou COMO fazer o que foi explicado",
    "pergunta curta sobre o PRAZO ou o TEMPO, somente se o trecho trouxer essa informação",
    "pergunta curta sobre o que GUARDAR ou levar como prova",
    "pergunta curta de esclarecimento sobre algo que o atendente acabou de dizer",
    "pergunta curta sobre o que acontece se a regra NÃO for cumprida",
    "pergunta curta sobre QUEM é o responsável",
]
ANGLES = ["", "A situação envolve uma compra feita pelo celular.",
          "O consumidor escreve de forma bem simples e direta, como numa mensagem de WhatsApp.",
          "O problema é com um serviço (banco, telefone ou internet), e não com um produto.",
          "O consumidor é uma pessoa mais velha, com pouca prática com internet."]


def dialogue_messages(chunk, topic, angle="", style=""):
    user = (
        f"Documento: {chunk['titulo']}\nSeção: {chunk['secao']}\n\nTrecho:\n\"\"\"\n{chunk['text']}\n\"\"\"\n\n"
        f"Tema: {topic}. {angle}\nCrie um diálogo curto entre um consumidor e um atendente de defesa do consumidor, com 2 trocas.\n"
        "Regras:\n"
        "1. A primeira pergunta do consumidor soa natural, em português simples.\n"
        f"2. A segunda pergunta do consumidor é uma {style}. Ela só faz sentido depois da primeira resposta, é curta "
        "e NÃO pode ser 'E se eles não resolverem?'.\n"
        "3. As respostas do atendente usam SOMENTE informações do trecho, em até 3 frases curtas, em português correto, "
        "linguagem simples e educada. Responda exatamente ao que foi perguntado em cada troca. Se a regra vale só para "
        "certos fornecedores, diga para quem vale.\n"
        "4. Não invente números, nomes, prazos ou telefones.\n"
        'Formato: {"turnos": [{"usuario": "...", "atendente": "..."}, {"usuario": "...", "atendente": "..."}]}')
    return [{"role": "system", "content": QA_SYSTEM}, {"role": "user", "content": user}]


def dependency_messages(first_user, first_assistant, second_user):
    user = (f"Pergunta 1: {first_user}\nResposta 1: {first_assistant}\nPergunta 2: {second_user}\n\n"
            "A pergunta 2 só faz sentido depois da primeira troca, isto é, depende do contexto da conversa? "
            "Responda apenas SIM ou NÃO.")
    return [{"role": "system", "content": "Você é um revisor rigoroso de diálogos."}, {"role": "user", "content": user}]


def chat_judge_messages(kind, chunk_text, turns):
    """Juízes do diálogo: fidelidade ao trecho, resposta pertinente em cada troca e português correto."""
    dialogue = "\n".join(f"Consumidor: {user}\nAtendente: {assistant}" for user, assistant in turns)
    if kind == "fid":
        task = ("As respostas do atendente usam SOMENTE informações do trecho? Indicar o Procon, o consumidor.gov.br, a "
                "Defensoria Pública ou um advogado é permitido mesmo sem estar no trecho, mas qualquer outro fato, regra, "
                "exceção, prazo ou valor que não esteja no trecho exige a resposta NÃO. Responda apenas SIM ou NÃO.")
    elif kind == "rel":
        task = ("Cada resposta do atendente responde ao que o consumidor perguntou naquela troca, sem trocar de assunto "
                "e sem afirmar o contrário do que o trecho diz? Responda apenas SIM ou NÃO.")
    else:
        task = ("As respostas do atendente estão em português correto (concordância e ortografia) e em linguagem "
                "simples? Responda apenas SIM ou NÃO.")
    user = f"Trecho:\n\"\"\"\n{chunk_text}\n\"\"\"\n\nDiálogo:\n{dialogue}\n\n{task}"
    return [{"role": "system", "content": "Você é um revisor rigoroso de dados de treino."}, {"role": "user", "content": user}]


# Algumas saídas do modelo põem os dois turnos dentro de um único objeto, repetindo as chaves; o json.loads
# guardaria só o último par. Por isso extraímos os pares "usuario"/"atendente" em ordem, direto do texto.
TURN_RE = re.compile(r'"usuario"\s*:\s*"((?:[^"\\]|\\.)*)"\s*,\s*"atendente"\s*:\s*"((?:[^"\\]|\\.)*)"', re.S)


def extract_turns(output):
    turns = []
    for user, assistant in TURN_RE.findall(output):
        try:
            turns.append((json.loads(f'"{user}"'), json.loads(f'"{assistant}"')))
        except json.JSONDecodeError:
            turns.append((user, assistant))
    return turns


def validate_dialogue(turns, chunk_text):
    """Devolve (turnos, motivo_da_rejeicao). Os limites de tamanho são só de formato; a fidelidade é dos juízes."""
    if len(turns) < 2:
        return None, "menos de 2 turnos"
    cleaned = []
    for user, assistant in turns[:2]:
        user, assistant = user.strip(), assistant.strip()
        if not (5 <= len(user) <= 250) or not (20 <= len(assistant) <= 900):
            return None, "tamanho fora do limite"
        if REFERENCE_RE.search(assistant):
            return None, "cita artigo/lei (o consumidor não fala assim)"
        if not numbers_in(assistant) <= numbers_in(chunk_text):
            return None, "número fora do trecho"
        cleaned.append((user, assistant))
    if len(cleaned[1][0].split()) > 18:
        return None, "2ª pergunta longa demais"
    if norm_key(cleaned[1][0]) in {"e se eles nao resolverem", "e se eles nao resolverem?"}:
        return None, "2ª pergunta genérica repetida"
    return cleaned, ""


chunk_vectors = embed([row["text"] for row in text_rows])
topic_rankings = {topic: [int(i) for i in np.argsort(-(chunk_vectors @ embed([query])[0]))] for topic, _, query in TOPICS}
NEEDED_GENERATED = TARGET_CHAT - len(MANUAL_DIALOGUES)
MAX_SAME_SECOND_QUESTION = 2

CHAT_STATS, REJECTED_STRUCTURE = Counter(), Counter()
acceptable, kept_vectors, second_questions = [], [], Counter()
CHAT_JUDGES = {"fid": "juiz: informação fora do trecho", "rel": "juiz: resposta não pertinente", "lang": "juiz: português incorreto"}


def run_dialogue_round(round_index):
    """Gera, valida e julga uma rodada de diálogos. Cada rodada usa um ângulo e um estilo de 2ª pergunta diferentes
    (um trecho pode aparecer em temas e rodadas diferentes: o prompt muda)."""
    angle = ANGLES[round_index]
    slots = [(topic, quota, index) for topic, quota, _ in TOPICS for index in topic_rankings[topic][:quota * 3]]
    styles = [SECOND_TURN_STYLES[(index + round_index + position) % len(SECOND_TURN_STYLES)]
              for position, (_, _, index) in enumerate(slots)]
    ids = [short_hash("chat2", text_rows[i]["text"], topic, angle, style) for (topic, _, i), style in zip(slots, styles)]
    jobs = [(job_id, dialogue_messages(text_rows[i], topic, angle, style))
            for job_id, (topic, _, i), style in zip(ids, slots, styles)]
    outputs = run_llm_jobs(f"chat_dialogos_v2_r{round_index}", jobs, max_new_tokens=700, batch_size=6)
    candidates = []
    for job_id, (topic, quota, index) in zip(ids, slots):
        turns, reason = validate_dialogue(extract_turns(outputs[job_id]), text_rows[index]["text"])
        if turns:
            candidates.append({"topic": topic, "quota": quota, "chunk": index, "turns": turns})
        else:
            REJECTED_STRUCTURE[reason] += 1
    print(f"Rodada {round_index + 1}: {len(candidates)} de {len(slots)} válidos na estrutura")
    verdicts = {}
    for kind in CHAT_JUDGES:
        judge_ids = [short_hash("v2", kind, c["chunk"], c["turns"][0][0]) for c in candidates]
        judge_jobs = [(job_id, chat_judge_messages(kind, text_rows[c["chunk"]]["text"], c["turns"]))
                      for job_id, c in zip(judge_ids, candidates)]
        judged = run_llm_jobs(f"chat_juiz_v2_{kind}", judge_jobs, max_new_tokens=4, batch_size=8)
        verdicts[kind] = [is_yes(judged[job_id]) for job_id in judge_ids]
    dependency_ids = [short_hash("v2", "dep", c["chunk"], c["turns"][0][0]) for c in candidates]
    dependency_jobs = [(job_id, dependency_messages(c["turns"][0][0], c["turns"][0][1], c["turns"][1][0]))
                       for job_id, c in zip(dependency_ids, candidates)]
    dependency_out = run_llm_jobs("chat_juiz_v2_dep", dependency_jobs, max_new_tokens=4, batch_size=8)
    approved = []
    for position, candidate in enumerate(candidates):
        failed = [kind for kind in CHAT_JUDGES if not verdicts[kind][position]]
        if failed:
            CHAT_STATS[f"rejeitados: {CHAT_JUDGES[failed[0]]}"] += 1
        elif not is_yes(dependency_out[dependency_ids[position]]):
            CHAT_STATS["rejeitados: 2ª pergunta não depende da 1ª"] += 1
        else:
            approved.append(candidate)
    # Sem vazamento do teste, sem diálogos repetidos entre si e sem a mesma 2ª pergunta muitas vezes.
    vectors = embed([c["turns"][0][0] for c in approved]) if approved else []
    for candidate, vector in zip(approved, vectors):
        second_key = norm_key(candidate["turns"][1][0])
        if float((test_embeddings @ vector).max()) >= LEAK_THRESHOLD:
            CHAT_STATS["removidos: parecidos com o teste"] += 1
        elif kept_vectors and float((np.array(kept_vectors) @ vector).max()) >= NEAR_DUP_THRESHOLD:
            CHAT_STATS["quase-duplicatas removidas"] += 1
        elif second_questions[second_key] >= MAX_SAME_SECOND_QUESTION:
            CHAT_STATS["removidos: 2ª pergunta repetida"] += 1
        else:
            second_questions[second_key] += 1
            acceptable.append(candidate)
            kept_vectors.append(vector)


# Repete com ângulos novos até ter diálogos suficientes (com folga para a cota por tema).
for round_index in range(len(ANGLES)):
    run_dialogue_round(round_index)
    print(f"    aprovados até agora: {len(acceptable)} (precisamos de {NEEDED_GENERATED})")
    if len(acceptable) >= NEEDED_GENERATED + 4:
        break

generated_dialogues, per_topic = [], Counter()
for candidate in acceptable:
    if per_topic[candidate["topic"]] < candidate["quota"]:
        per_topic[candidate["topic"]] += 1
        generated_dialogues.append(candidate)
for candidate in acceptable:     # se faltar, completa com os aprovados que sobraram
    if len(generated_dialogues) >= NEEDED_GENERATED:
        break
    if candidate not in generated_dialogues:
        per_topic[candidate["topic"]] += 1
        generated_dialogues.append(candidate)
print(f"Diálogos gerados aprovados: {len(generated_dialogues)} | por tema: {dict(per_topic)}")
print("Rejeições de estrutura:", dict(REJECTED_STRUCTURE))
for reason, count in sorted(CHAT_STATS.items()):
    print(f"    {reason}: {count}")
assert len(generated_dialogues) + len(MANUAL_DIALOGUES) >= 30, "Poucos diálogos aprovados: veja as rejeições acima."

[chat_dialogos_v2_r0] 6/66 novas (39s)
[chat_dialogos_v2_r0] 36/66 novas (227s)
[chat_dialogos_v2_r0] 66/66 novas (416s)
Rodada 1: 42 de 66 válidos na estrutura
[chat_juiz_v2_fid] 8/42 novas (6s)
[chat_juiz_v2_fid] 42/42 novas (31s)
[chat_juiz_v2_rel] 8/42 novas (6s)
[chat_juiz_v2_rel] 42/42 novas (29s)
[chat_juiz_v2_lang] 8/42 novas (6s)
[chat_juiz_v2_lang] 42/42 novas (28s)
[chat_juiz_v2_dep] 8/42 novas (2s)
[chat_juiz_v2_dep] 42/42 novas (8s)
    aprovados até agora: 22 (precisamos de 22)
[chat_dialogos_v2_r1] 6/66 novas (34s)
[chat_dialogos_v2_r1] 36/66 novas (230s)
[chat_dialogos_v2_r1] 66/66 novas (424s)
Rodada 2: 36 de 66 válidos na estrutura
[chat_juiz_v2_fid] 8/35 novas (6s)
[chat_juiz_v2_rel] 8/35 novas (6s)
[chat_juiz_v2_lang] 8/35 novas (6s)
[chat_juiz_v2_dep] 8/35 novas (2s)
    aprovados até agora: 42 (precisamos de 22)
Diálogos gerados aprovados: 22 | por tema: {'defeito e garantia': 6, 'compra online e arrependimento': 4, 'cobrança indevida e cadastro de inadimplentes':

### 10.2 Montagem e gravação

Juntamos os diálogos escritos à mão e os gerados, "sujamos" uma parte das perguntas do usuário
(as respostas do atendente ficam intactas), embaralhamos com semente fixa e gravamos.

In [23]:
rng_chat = random.Random(SEED + 2)
entries = []   # (tipo, fonte, [(usuário, atendente), ...])
for kind, source, turns in MANUAL_DIALOGUES:
    entries.append((kind, source, list(turns)))
for candidate in generated_dialogues:
    turns = [(make_informal(user, rng_chat) if rng_chat.random() < INFORMAL_RATE else user, assistant)
             for user, assistant in candidate["turns"]]
    entries.append(("normal_llm", text_rows[candidate["chunk"]]["fonte"], turns))
rng_chat.shuffle(entries)

chat_rows, CHAT_KINDS = [], []
for kind, source, turns in entries:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for user, assistant in turns:
        messages += [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]
    chat_rows.append({"messages": messages, "fonte": source})
    CHAT_KINDS.append(kind)

CHAT_PATH = OUT_DIR / "corpus_chat.jsonl"
write_jsonl(CHAT_PATH, chat_rows)
refusals = sum(kind in REFUSAL_KINDS for kind in CHAT_KINDS)
user_turns = [m["content"] for row in chat_rows for m in row["messages"] if m["role"] == "user"]
chat_informal = sum(looks_informal(text) for text in user_turns)
print(f"corpus_chat.jsonl: {len(chat_rows)} diálogos (mínimo: 30) | por tipo: {dict(Counter(CHAT_KINDS))}")
print(f"Recusas e encaminhamentos: {refusals} (mínimo: 10) | perguntas do usuário com língua real: "
      f"{chat_informal}/{len(user_turns)} ({chat_informal / len(user_turns):.0%})")
assert len(chat_rows) >= 30 and refusals >= 10
assert all(sum(m["role"] == "user" for m in row["messages"]) >= 2 for row in chat_rows)

corpus_chat.jsonl: 36 diálogos (mínimo: 30) | por tipo: {'urgencia': 4, 'normal_llm': 22, 'fora_do_tema': 4, 'normal_manual': 2, 'orientacao_individual': 4}
Recusas e encaminhamentos: 12 (mínimo: 10) | perguntas do usuário com língua real: 42/72 (58%)


## 11. Relatório de qualidade

Estatísticas de cada corpus (exemplos, média de tokens, duplicatas removidas, recusas e
encaminhamentos), cinco exemplos aleatórios, conferência de que **nenhuma pergunta do teste aparece
no treino**, nova verificação de dados pessoais nos corpus finais, taxa de língua real e as
planilhas para a revisão humana (`revisao.csv` e `amostra_fidelidade.csv`).

In [24]:
FINAL = {
    "corpus_texto.jsonl": read_jsonl(TEXT_PATH),
    "corpus_qa.jsonl": read_jsonl(QA_PATH),
    "corpus_chat.jsonl": read_jsonl(CHAT_PATH),
    "corpus_teste.jsonl": read_jsonl(TEST_PATH),
}


def chat_text(row):
    return "\n".join(message["content"] for message in row["messages"])


def user_turns_of(row):
    return [message["content"] for message in row["messages"] if message["role"] == "user"]


def is_referral(text):
    lowered = text.lower()
    return any(marker in lowered for marker in ("só ajudo", "192", "193", "defensoria", "advogado", "procure"))


qa_dups = sum(count for key, count in QA_STATS.items() if key.startswith(("duplicatas", "quase")))
duplicates_removed = {
    "corpus_texto.jsonl": sum(TEXT_DUPLICATES.values()),
    "corpus_qa.jsonl": qa_dups,
    "corpus_chat.jsonl": sum(count for key, count in CHAT_STATS.items() if key.startswith(("quase", "duplicatas"))),
    "corpus_teste.jsonl": 0,
}
mean_tokens = {
    "corpus_texto.jsonl": np.mean([count_tokens(row["text"]) for row in FINAL["corpus_texto.jsonl"]]),
    "corpus_qa.jsonl": np.mean([count_tokens(row["instruction"] + " " + row["output"]) for row in FINAL["corpus_qa.jsonl"]]),
    "corpus_chat.jsonl": np.mean([count_tokens(chat_text(row)) for row in FINAL["corpus_chat.jsonl"]]),
    "corpus_teste.jsonl": np.mean([count_tokens(row["pergunta"] + " " + row["resposta_esperada"]) for row in FINAL["corpus_teste.jsonl"]]),
}
refusals_by_corpus = {
    "corpus_texto.jsonl": 0,
    "corpus_qa.jsonl": sum(is_referral(row["output"]) for row in FINAL["corpus_qa.jsonl"]),
    "corpus_chat.jsonl": sum(kind in REFUSAL_KINDS for kind in CHAT_KINDS),
    "corpus_teste.jsonl": sum(row["tipo"] in ("fora_do_tema", "urgencia") for row in FINAL["corpus_teste.jsonl"]),
}

print(f"{'corpus':22} {'exemplos':>8} {'média de tokens':>16} {'duplicatas removidas':>21} {'recusas/encaminh.':>18}")
for name, rows in FINAL.items():
    print(f"{name:22} {len(rows):8} {mean_tokens[name]:16.0f} {duplicates_removed[name]:21} {refusals_by_corpus[name]:18}")

corpus                 exemplos  média de tokens  duplicatas removidas  recusas/encaminh.
corpus_texto.jsonl           68              461                     4                  0
corpus_qa.jsonl             126               45                    65                  2
corpus_chat.jsonl            36              281                     7                 12
corpus_teste.jsonl           24               52                     0                  7


### Cinco exemplos aleatórios de cada corpus

In [25]:
rng_samples = random.Random(SEED + 3)
for name, rows in FINAL.items():
    print("=" * 20, name)
    for row in rng_samples.sample(rows, 5):
        if name == "corpus_texto.jsonl":
            print(f"• [{row['fonte']} | {row['secao']} | {row['data']}] {row['text'][:170].replace(chr(10), ' ')}…")
        elif name == "corpus_qa.jsonl":
            print(f"• P: {row['instruction']}\n  R: {row['output']}\n  trecho: {row['trecho'][:120]}…")
        elif name == "corpus_chat.jsonl":
            dialogue = " | ".join(f"{m['role'][0].upper()}: {m['content']}" for m in row["messages"][1:])
            print(f"• {dialogue[:330]}")
        else:
            print(f"• ({row['tipo']}) {row['pergunta']}  →  {row['resposta_esperada'][:110]}")

==================== corpus_texto.jsonl
• [cdc_lei_8078_1990.html | Tít. III, Cap. II – Das Ações Coletivas Para a Defesa de Interesses Individuais Homogêneos (arts. 91 a 98) | 1990-09] Art. 91. Os legitimados de que trata o art. 82 poderão propor, em nome próprio e no interesse das vítimas ou seus sucessores, ação civil coletiva de responsabilidade pelo…
• [decreto_13135_2026.docx | Cap. II – Da qualidade do tratamento das demandas (arts. 4 a 9) | 2026-10] Art. 4º No tratamento das demandas, o SAC garantirá: I - a tempestividade; II - a segurança; III - a privacidade; IV - a eficiência; V - a celeridade; VI - a cordialidade…
• [decreto_7962_2013.txt | Decreto nº 7.962/2013 — Comércio eletrônico (arts. 3 a 5) | 2013-03] II - prazo para utilização da oferta pelo consumidor; e III - identificação do fornecedor responsável pelo sítio eletrônico e do fornecedor do produto ou serviço ofertado…
• [cdc_lei_8078_1990.html | Tít. III, Cap. I – Disposições Gerais (arts. 82 a 84) | 1990-09] II - 

### Sem duplicatas e sem vazamento do teste

Conferimos nos arquivos **finais**: duplicatas exatas (depois da normalização) e quase-duplicatas
entre as perguntas, e a similaridade de cada pergunta de treino com as do teste.

In [26]:
qa_questions = [row["instruction"] for row in FINAL["corpus_qa.jsonl"]]
chat_first_questions = [user_turns_of(row)[0] for row in FINAL["corpus_chat.jsonl"]]
all_train_questions = qa_questions + [t for row in FINAL["corpus_chat.jsonl"] for t in user_turns_of(row)]

exact_text = len({dedup_key(row["text"]) for row in FINAL["corpus_texto.jsonl"]}) == len(FINAL["corpus_texto.jsonl"])
exact_qa = len({norm_key(q) for q in qa_questions}) == len(qa_questions)
exact_chat = len({norm_key(q) for q in chat_first_questions}) == len(chat_first_questions)
print("Duplicatas exatas restantes — texto:", not exact_text, "| qa:", not exact_qa, "| chat:", not exact_chat)

train_vectors = embed(all_train_questions)
similarity_matrix = train_vectors @ test_embeddings.T
max_similarity = float(similarity_matrix.max())
worst = np.unravel_index(similarity_matrix.argmax(), similarity_matrix.shape)
exact_leaks = [q for q in all_train_questions if norm_key(q) in test_keys]
print(f"Perguntas de treino comparadas com as do teste: {len(all_train_questions)}")
print(f"Vazamento exato: {len(exact_leaks)} | maior similaridade com o teste: {max_similarity:.2f} "
      f"(limite {LEAK_THRESHOLD})")
print(f"  par mais próximo → treino: {all_train_questions[worst[0]]!r}\n                    teste : {test_rows[worst[1]]['pergunta']!r}")
assert exact_text and exact_qa and exact_chat, "Há duplicatas exatas nos arquivos finais"
assert not exact_leaks and max_similarity < LEAK_THRESHOLD, "Há pergunta do teste vazando para o treino"
assert hashlib.sha256(TEST_PATH.read_bytes()).hexdigest() == TEST_SHA256, "O conjunto de teste foi alterado"
print("Teste intacto (SHA-256 confere) e sem vazamento.")

Duplicatas exatas restantes — texto: False | qa: False | chat: False
Perguntas de treino comparadas com as do teste: 198
Vazamento exato: 0 | maior similaridade com o teste: 0.85 (limite 0.88)
  par mais próximo → treino: 'Eu posso escolher como receber a resposta da empresa sobre minha reclamação?'
                    teste : 'prazo pra empresa responder minha reclamacao no sac?'
Teste intacto (SHA-256 confere) e sem vazamento.


### Dados pessoais nos corpus finais e língua real

In [27]:
fields = []
for row in FINAL["corpus_texto.jsonl"]:
    fields.append(row["text"])
for row in FINAL["corpus_qa.jsonl"]:
    fields += [row["instruction"], row["output"], row["trecho"]]
for row in FINAL["corpus_chat.jsonl"]:
    fields += [message["content"] for message in row["messages"]]
for row in FINAL["corpus_teste.jsonl"]:
    fields += [row["pergunta"], row["resposta_esperada"]]
_, final_pii, _ = anonymize_texts(fields)
final_pii = {kind: total for kind, total in final_pii.items() if total}
print("Dados pessoais encontrados nos corpus finais:", final_pii or "nenhum")
assert not {"CPF", "EMAIL", "TELEFONE"} & set(final_pii), "Sobrou CPF, e-mail ou telefone nos corpus"
if "NOME" in final_pii:   # o NER pode errar; avisamos em vez de interromper
    print("ATENÇÃO: o detector de nomes marcou", final_pii["NOME"], "trecho(s) nos corpus finais; revise manualmente.")

informal = {
    "corpus_qa.jsonl": [looks_informal(q) for q in qa_questions],
    "corpus_chat.jsonl": [looks_informal(t) for row in FINAL["corpus_chat.jsonl"] for t in user_turns_of(row)],
    "corpus_teste.jsonl": [looks_informal(row["pergunta"]) for row in FINAL["corpus_teste.jsonl"]],
}
for name, flags in informal.items():
    print(f"Língua real em {name}: {sum(flags)}/{len(flags)} ({sum(flags) / len(flags):.0%}) — mínimo exigido: 30%")
    assert sum(flags) / len(flags) >= 0.30

Dados pessoais encontrados nos corpus finais: nenhum
Língua real em corpus_qa.jsonl: 48/126 (38%) — mínimo exigido: 30%
Língua real em corpus_chat.jsonl: 42/72 (58%) — mínimo exigido: 30%
Língua real em corpus_teste.jsonl: 12/24 (50%) — mínimo exigido: 30%


### Planilhas para a revisão humana

- `revisao.csv` (colunas `arquivo, linha, alterado, comentário`): amostra de **25%** dos exemplos
  gerados pelo modelo (acima dos 20% exigidos). A coluna `linha` é a linha do arquivo `.jsonl`.
- `revisao_apoio_qa.csv` e `revisao_apoio_chat.csv`: o **texto** de cada exemplo sorteado, para o grupo
  revisar numa planilha (separador `;`). Em cada linha, preencher `decisao` (ok, corrigir ou excluir), o texto
  corrigido quando for o caso e o `comentario`. Depois, o script `preparacao/aplicar_revisao.py` aplica as
  correções nos `.jsonl` e grava o `revisao.csv` final (arquivo, linha, alterado, comentário).
- `amostra_fidelidade.csv`: 20 pares de perguntas e respostas aleatórios para conferir se a resposta
  é fiel ao `trecho`. O grupo preenche `fiel_humano` (sim/não); o relatório informa quantos dos 20
  estavam fiéis. A coluna `sobreposicao` é só um apoio (fração das palavras da resposta que estão no trecho).

In [28]:
import csv


def write_csv(path, header, rows):
    with open(path, "w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(header)
        writer.writerows(rows)


rng_review = random.Random(SEED + 4)
qa_lines = list(range(1, len(FINAL["corpus_qa.jsonl"]) + 1))
generated_chat_lines = [i for i, kind in enumerate(CHAT_KINDS, start=1) if kind == "normal_llm"]
review_rows = []
for filename, lines in (("corpus_qa.jsonl", qa_lines), ("corpus_chat.jsonl", generated_chat_lines)):
    size = -(-len(lines) // 4)                       # 25%, arredondando para cima
    review_rows += [(filename, line, "", "") for line in sorted(rng_review.sample(lines, size))]
write_csv(OUT_DIR / "revisao.csv", ["arquivo", "linha", "alterado", "comentário"], review_rows)
generated_total = len(qa_lines) + len(generated_chat_lines)
print(f"revisao.csv: {len(review_rows)} de {generated_total} exemplos gerados pelo modelo "
      f"({len(review_rows) / generated_total:.0%}; mínimo: 20%)")


def write_review_sheets():
    """Planilhas de apoio (separador ';', que o Excel em português abre direto) com o texto de cada exemplo
    sorteado, para o grupo revisar sem abrir os .jsonl. O script aplicar_revisao.py devolve as correções aos arquivos."""
    qa_header = ["arquivo", "linha", "pergunta", "resposta", "trecho", "fonte", "decisao (ok/corrigir/excluir)",
                 "nova_pergunta", "nova_resposta", "comentario"]
    chat_header = ["arquivo", "linha", "usuario_1", "atendente_1", "usuario_2", "atendente_2", "decisao (ok/corrigir/excluir)",
                   "novo_atendente_1", "novo_atendente_2", "comentario"]
    qa_sheet, chat_sheet = [], []
    for filename, line, _, _ in review_rows:
        if filename == "corpus_qa.jsonl":
            row = FINAL[filename][line - 1]
            qa_sheet.append([filename, line, row["instruction"], row["output"], row["trecho"], row["fonte"], "", "", "", ""])
        else:
            turns = [m["content"] for m in FINAL[filename][line - 1]["messages"] if m["role"] != "system"]
            chat_sheet.append([filename, line, *turns, "", "", "", ""])
    for name, header, sheet in (("revisao_apoio_qa.csv", qa_header, qa_sheet), ("revisao_apoio_chat.csv", chat_header, chat_sheet)):
        with open(OUT_DIR / name, "w", encoding="utf-8-sig", newline="") as handle:
            writer = csv.writer(handle, delimiter=";")
            writer.writerow(header)
            writer.writerows(sheet)
    print(f"revisao_apoio_qa.csv: {len(qa_sheet)} linhas | revisao_apoio_chat.csv: {len(chat_sheet)} linhas")


write_review_sheets()


def support_overlap(answer, support):
    answer_words = set(re.findall(r"\w{4,}", answer.lower()))
    support_words = set(re.findall(r"\w{4,}", support.lower()))
    return len(answer_words & support_words) / max(1, len(answer_words))


sample_lines = sorted(rng_review.sample(qa_lines, 20))
fidelity_rows = []
for line in sample_lines:
    row = FINAL["corpus_qa.jsonl"][line - 1]
    fidelity_rows.append((line, row["instruction"], row["output"], row["trecho"], row["fonte"],
                          f"{support_overlap(row['output'], row['trecho']):.2f}", "", ""))
write_csv(OUT_DIR / "amostra_fidelidade.csv",
          ["linha", "pergunta", "resposta", "trecho", "fonte", "sobreposicao", "fiel_humano", "observacao"],
          fidelity_rows)
print("amostra_fidelidade.csv: 20 pares para conferir (preencher fiel_humano).")

revisao.csv: 38 de 148 exemplos gerados pelo modelo (26%; mínimo: 20%)
revisao_apoio_qa.csv: 32 linhas | revisao_apoio_chat.csv: 6 linhas
amostra_fidelidade.csv: 20 pares para conferir (preencher fiel_humano).


In [29]:
# Resumo para o relatório em PDF (estatísticas, anonimização e limpeza) e metadados dos documentos.
summary = {
    "corpus": {name: {"exemplos": len(rows), "media_tokens": round(float(mean_tokens[name]), 1),
                      "duplicatas_removidas": duplicates_removed[name], "recusas_encaminhamentos": refusals_by_corpus[name]}
               for name, rows in FINAL.items()},
    "qa_filtros": dict(QA_STATS), "chat_filtros": dict(CHAT_STATS),
    "vazamento_maior_similaridade": round(max_similarity, 3), "teste_sha256": TEST_SHA256,
    "lingua_real": {name: round(sum(flags) / len(flags), 3) for name, flags in informal.items()},
    "modelos": {"geracao": GENERATOR_NAME, "tokenizador": TOKENIZER_NAME, "embeddings": EMBEDDING_NAME, "ner": SPACY_MODEL},
}
(OUT_DIR / "relatorio_qualidade.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
(OUT_DIR / "metadados_documentos.json").write_text(json.dumps(
    {"documentos": DOCS, "limpeza": {n: dict(c) for n, c in CLEAN_STATS.items()}, "anonimizacao": ANON_REPORT},
    ensure_ascii=False, indent=1), encoding="utf-8")
(OUT_DIR / "anonimizacao_antes_depois.txt").write_text(
    "ANTES\n=====\n" + ficha_text + "\n\nDEPOIS\n======\n" + ficha_after[0], encoding="utf-8")
print("Gravados: relatorio_qualidade.json, metadados_documentos.json, anonimizacao_antes_depois.txt")

Gravados: relatorio_qualidade.json, metadados_documentos.json, anonimizacao_antes_depois.txt


## 12. Interface simples (Gradio)

Demonstração para o vídeo: o usuário envia arquivos (PDF, TXT, DOCX ou HTML) e/ou cola um texto, e a
ferramenta mostra o resultado de **extração → limpeza → normalização → anonimização → segmentação**.
A geração dos corpus com o modelo está nas seções 9 e 10 (usa a GPU e leva mais tempo).

Para abrir a interface, mude `RUN_GRADIO` para `True` e execute a célula. Na execução gravada
deste notebook ela fica desligada para não bloquear o "Executar tudo".

In [30]:
RUN_GRADIO = False


def process_documents(uploaded_files, pasted_text, structure):
    """Executa o pipeline sobre arquivos enviados e/ou texto colado e devolve um resumo e os trechos."""
    sources = []
    for item in uploaded_files or []:
        path = Path(getattr(item, "name", item))
        shutil.copy(path, INPUT_DIR / path.name)
        sources.append(path.name)
    if pasted_text and pasted_text.strip():
        sources.append(ingest_pasted_text("texto_colado_interface.txt", pasted_text, {}))
    if not sources:
        return "Envie um arquivo ou cole um texto.", ""
    report, previews = [], []
    for name in sources:
        DOCS[name] = {"title": name, "agency": "informar", "link": "n/a", "date": time.strftime("%Y-%m"),
                      "accessed": time.strftime("%Y-%m-%d"), "structure": structure, "is_test": False, "volatile": "informar"}
        path = INPUT_DIR / name
        lines, whole = EXTRACTORS[path.suffix.lower()](path)
        units, stats = clean_document(name, {"lines": lines, "whole_paragraphs": whole})
        for unit in units:
            unit["text"] = normalize_text(unit["text"])
        anonymized, counts, _ = anonymize_texts([unit["text"] for unit in units])
        for unit, text in zip(units, anonymized):
            unit["text"] = text
        UNITS[name] = units
        chunks = build_chunks(name)
        masked = {kind: total for kind, total in counts.items() if total}
        report.append(f"{name}: {len(lines)} linhas → {len(units)} parágrafos → {len(chunks)} trechos | "
                      f"limpeza: {dict(stats)} | mascarados: {masked or 'nenhum'}")
        previews += [f"[{c['fonte']} | {c['secao']} | {c['tokens']} tokens]\n{c['text'][:500]}" for c in chunks[:3]]
    return "\n".join(report), "\n\n".join(previews)


if RUN_GRADIO:
    pip_install(["gradio"])
    import gradio as gr
    demo = gr.Interface(
        fn=process_documents,
        inputs=[gr.File(file_count="multiple", file_types=[".pdf", ".txt", ".docx", ".html"], label="Documentos"),
                gr.Textbox(lines=6, label="Ou cole um texto"),
                gr.Radio(["legal", "guia", "ficha"], value="legal", label="Tipo de documento")],
        outputs=[gr.Textbox(label="Resumo do processamento"), gr.Textbox(lines=14, label="Primeiros trechos (anonimizados)")],
        title="Fábrica de Corpus CitizenTech — pré-processamento", allow_flagging="never")
    demo.launch(share=True)
else:
    summary_text, preview_text = process_documents([], "Contato do consumidor Pedro Alves, CPF 529.982.247-25, e-mail pedro.alves@exemplo.com.", "ficha")
    print("Teste rápido da função da interface (texto colado fictício):\n", summary_text)

Teste rápido da função da interface (texto colado fictício):
 texto_colado_interface.txt: 1 linhas → 1 parágrafos → 1 trechos | limpeza: {'caracteres estranhos removidos': 0, 'notas de alteração removidas': 0, 'reticências removidas': 0, 'URLs removidas': 0} | mascarados: {'EMAIL': 1, 'CPF': 1, 'NOME': 1}


## 13. Conferência final, arquivos gerados e download

In [31]:
checks = [
    ("Ingestão de PDF, HTML, DOCX, TXT e texto colado", {Path(n).suffix for n in DOCS} >= {".pdf", ".html", ".docx", ".txt"}),
    ("3 a 8 documentos oficiais", 3 <= sum(not m["is_test"] for m in DOCS.values()) <= 8),
    ("Documento-teste fictício fora dos corpus", all(DOCS[r["fonte"]]["is_test"] is False
                                                   for r in FINAL["corpus_texto.jsonl"])),
    ("corpus_texto.jsonl com 50 trechos ou mais", len(FINAL["corpus_texto.jsonl"]) >= 50),
    ("corpus_qa.jsonl com 100 pares ou mais, de 3 documentos ou mais",
     len(FINAL["corpus_qa.jsonl"]) >= 100 and len({r["fonte"] for r in FINAL["corpus_qa.jsonl"]}) >= 3),
    ("corpus_chat.jsonl com 30 diálogos e 10 recusas ou mais",
     len(FINAL["corpus_chat.jsonl"]) >= 30 and refusals_by_corpus["corpus_chat.jsonl"] >= 10),
    ("corpus_teste.jsonl com 20 perguntas, 3 fora do tema e 2 de urgência",
     len(test_rows) >= 20 and type_counts["fora_do_tema"] >= 3 and type_counts["urgencia"] >= 2),
    ("Trechos entre 256 e 512 tokens", all(MIN_TOKENS <= count_tokens(r["text"]) <= MAX_TOKENS for r in FINAL["corpus_texto.jsonl"])),
    ("Sem vazamento do teste no treino", max_similarity < LEAK_THRESHOLD and not exact_leaks),
    ("Sem CPF, e-mail ou telefone nos corpus finais", not ({"CPF", "EMAIL", "TELEFONE"} & set(final_pii))),
    ("Língua real em pelo menos 30% das perguntas", all(sum(f) / len(f) >= 0.30 for f in informal.values())),
    ("revisao.csv com pelo menos 20% dos exemplos gerados", len(review_rows) / generated_total >= 0.20),
]
for label, ok in checks:
    print(("[ok]  " if ok else "[FALHOU] ") + label)

print("\nArquivos gerados em", OUT_DIR)
for path in sorted(OUT_DIR.iterdir()):
    if path.is_file():
        print(f"  {path.name:34} {path.stat().st_size / 1024:8.1f} KB")

archive = shutil.make_archive(str(WORK_DIR / "corpus_gs_nlp"), "zip", OUT_DIR)
print("\nPacote com todos os arquivos:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

[ok]  Ingestão de PDF, HTML, DOCX, TXT e texto colado
[ok]  3 a 8 documentos oficiais
[ok]  Documento-teste fictício fora dos corpus
[ok]  corpus_texto.jsonl com 50 trechos ou mais
[ok]  corpus_qa.jsonl com 100 pares ou mais, de 3 documentos ou mais
[ok]  corpus_chat.jsonl com 30 diálogos e 10 recusas ou mais
[ok]  corpus_teste.jsonl com 20 perguntas, 3 fora do tema e 2 de urgência
[ok]  Trechos entre 256 e 512 tokens
[ok]  Sem vazamento do teste no treino
[ok]  Sem CPF, e-mail ou telefone nos corpus finais
[ok]  Língua real em pelo menos 30% das perguntas
[ok]  revisao.csv com pelo menos 20% dos exemplos gerados

Arquivos gerados em /content/gs_nlp/saida
  amostra_fidelidade.csv                 11.2 KB
  anonimizacao_antes_depois.txt           2.8 KB
  corpus_chat.jsonl                      42.6 KB
  corpus_qa.jsonl                        73.7 KB
  corpus_teste.jsonl                      6.6 KB
  corpus_texto.jsonl                    139.3 KB
  metadados_documentos.json               

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
**Observações finais**

- O bônus (treino de um adapter LoRA e comparação A/B/C no conjunto de teste) **não** faz parte deste
  notebook.
- O relatório em PDF, o README e o vídeo acompanham o repositório. As estatísticas do relatório saem de
  `relatorio_qualidade.json`, e o exemplo de antes e depois da anonimização, de `anonimizacao_antes_depois.txt`.
- O `revisao.csv` e a `amostra_fidelidade.csv` devem ser revisados e preenchidos pelo grupo antes da entrega.